# CS781 Phase 5 — Notebook P5_02: Legal Statute Identification with the Phase-1 models

Multi-label: a document → the set of statute ids (100 statutes, ≈ 3.7 per document). One notebook runs a queue of
experiments (one per GPU) and writes, for each: dev scores (official macro-F1, micro-F1, subset accuracy, F1 per
frequency group), the decision rule chosen honestly on dev, an error analysis and a ready `submission.zip`.

| Step | What happens |
|---|---|
| 1 | config: Phase-1 weights, experiments, queues |
| 2 | inputs: P5_01's token files (memory-mapped), clean-up (no test copies in train or in dev tuning), Phase-1 checkpoints |
| 3 | model classes (NB3) + backbone and adaptation methods (P3_02) |
| 4 | **document representations**: first / last L tokens, or K chunks spread over the whole document |
| 5 | **heads**: pooled vector → linear(100), or **label-wise attention** (every statute attends to its own evidence) |
| 6 | **losses**: BCE, positive-weighted BCE, focal, **asymmetric loss** |
| 7 | **decision rules** (global / per-statute thresholds, at-least-one, top-k), **co-occurrence stacking**, metrics, error analysis |
| 8 | techniques: full, frozen, top-k, gradual unfreezing, BitFit, **LoRMA**, adapters, **prompt tuning**; smaller students; **KD** |
| 9 | **hierarchical two-stage model** (chunk embeddings → BiGRU → label attention over chunks) |
| 10 | unit tests | 11 | run the queue | 12 | results table |

**Kaggle settings**: Accelerator **GPU T4 x2**, Internet not needed.
**Inputs**: (1) the **P5_01 output** (`p5/data/…`), (2) the **Phase-1 weight datasets**, (3) for stage 4: the outputs of the
earlier P5_02 runs (as notebook outputs or Datasets — experiment folders are found at any depth).

In [1]:
# ============================================================================================
# SHARED SETUP  —  paste this SAME cell at the top of every Phase-1 notebook.
# Works on Kaggle (Save & Commit or interactive), Colab (Drive) and local machines.
# ============================================================================================
import os, sys, json, time, glob, shutil, random, math, re, zlib, hashlib, gc
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

SESSION_START = time.time()
TIME_BUDGET_H = 11.0   # Kaggle kills a session at 12h and then saves NOTHING -> every long loop stops itself before this

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
PROJECT   = "cs781_p1"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, SCRATCH = Path("/content/drive/MyDrive") / PROJECT, Path("/content/scratch")   # Drive = persistent
elif IN_KAGGLE:
    ROOT, SCRATCH = Path("/kaggle/working") / PROJECT, Path("/tmp/cs781_scratch")       # working = saved on commit
else:
    ROOT, SCRATCH = Path("./") / PROJECT, Path("./scratch")
ROOT    = Path(os.environ.get("CS781_ROOT", ROOT))
SCRATCH = Path(os.environ.get("CS781_SCRATCH", SCRATCH))
ROOT.mkdir(parents=True, exist_ok=True); SCRATCH.mkdir(parents=True, exist_ok=True)

ENV = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
print(f"env={ENV}  ROOT={ROOT}  SCRATCH={SCRATCH}")
for p in {ROOT, SCRATCH}:
    du = shutil.disk_usage(p); print(f"  disk @ {p}: free {du.free/1e9:.1f} GB / total {du.total/1e9:.1f} GB")


# ---------------------------------------------------------------- small utilities
def time_left_h():
    return TIME_BUDGET_H - (time.time() - SESSION_START) / 3600

def save_json(obj, path):
    """Atomic write: a crash mid-write can never leave a half-written (corrupt) state file."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, path)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists(): return default
    with open(path, encoding="utf-8") as f: return json.load(f)

def restore_from_kaggle_inputs(subdirs):
    """Kaggle 'Save & Commit' starts every run with an EMPTY /kaggle/working.
    To resume, attach the output of the previous version (Add Input -> Your Work -> Notebook output).
    This copies <PROJECT>/<subdir> from any attached input into ROOT (never overwriting newer local files)."""
    if not IN_KAGGLE: return
    for sub in subdirs:
        for cand in sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{sub}", recursive=True)):
            n = 0
            for src in Path(cand).rglob("*"):
                if src.is_file():
                    dst = ROOT / sub / src.relative_to(cand)
                    if not dst.exists():
                        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); n += 1
            print(f"restored {n} files: {cand} -> {ROOT/sub}")

def find_artifact(rel):
    """Look for ROOT/rel first, then inside any attached Kaggle input."""
    p = ROOT / rel
    if p.exists(): return p
    if IN_KAGGLE:
        hits = sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{rel}", recursive=True))
        if hits: return Path(hits[0])
    return None

def get_hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("HF")
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get("HF")
    except Exception as e:
        print("Could not read secret 'HF':", e)
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN
print("HF token found:", bool(HF_TOKEN))

# Optional: a PRIVATE HF dataset repo used only to move YOUR OWN artifacts between accounts/sessions.
HF_ARTIFACT_REPO = None   # e.g. "your-hf-username/cs781-p1-artifacts"

def hf_push(subdir):
    if not HF_ARTIFACT_REPO: print("HF_ARTIFACT_REPO not set; skip push"); return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_ARTIFACT_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(ROOT / subdir), path_in_repo=subdir,
                      repo_id=HF_ARTIFACT_REPO, repo_type="dataset")
    print("pushed", subdir)

def hf_pull(subdir):
    if not HF_ARTIFACT_REPO: return
    from huggingface_hub import snapshot_download
    snapshot_download(HF_ARTIFACT_REPO, repo_type="dataset", allow_patterns=[f"{subdir}/*", f"{subdir}/**"],
                      local_dir=str(ROOT), token=HF_TOKEN)
    print("pulled", subdir)


# ---------------------------------------------------------------- data utilities
# These MUST be identical in every notebook (the train/val split and cleaning depend on them).
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SOURCE_NAMES = ["acts", "high_courts", "supreme_court", "tribunals"]   # processing order (matters for dedup)
VAL_PER_10K  = 100            # 1% of documents of every source are held out (document-level split)
PROMPT_WORDS, CONT_WORDS = 512, 128

def extract_text(rec):
    if rec is None: return ""
    if isinstance(rec, str): return rec
    if isinstance(rec, list): return "\n".join(extract_text(r) for r in rec)
    if isinstance(rec, dict):
        for k in ("text", "content", "document", "judgment", "judgement", "body"):
            if k in rec: return extract_text(rec[k])
    return ""

def iter_docs(path):
    """Stream documents from a (possibly multi-GB) JSON file WITHOUT loading it all into RAM.
    Handles: [ {...}, ... ]   |   { id: {...}, ... }   |   JSON-lines."""
    import ijson
    with open(path, "rb") as f:
        head = f.read(1 << 16).lstrip()
        f.seek(0)
        if head[:1] == b"[":
            for rec in ijson.items(f, "item", use_float=True):
                yield extract_text(rec)
            return
        first_line = f.readline(50_000_000); second = f.read(1 << 10).lstrip()[:1]
        f.seek(0)
        is_jsonl = False
        if first_line.endswith(b"\n") and second == b"{":
            try: json.loads(first_line); is_jsonl = True
            except Exception: pass
        if is_jsonl:
            for line in f:
                if line.strip(): yield extract_text(json.loads(line))
        else:
            for _, rec in ijson.kvitems(f, "", use_float=True):
                yield extract_text(rec)

_ZW = re.compile(r"[​‌‍﻿­]")
_WS_ALL = re.compile(r"\s+")
_HSPACE = re.compile(r"[^\S\n]+")
_MANY_NL = re.compile(r"\n\s*\n\s*\n+")

def normalize(text, mode):
    """'collapse_all'  -> every whitespace run becomes one space (text looks like ' '.join(text.split()))
       'keep_newlines' -> collapse spaces/tabs, keep at most one blank line"""
    text = _ZW.sub("", text)
    if mode == "collapse_all":
        return _WS_ALL.sub(" ", text).strip()
    text = _HSPACE.sub(" ", text.replace("\r\n", "\n").replace("\r", "\n"))
    text = "\n".join(l.strip() for l in text.split("\n"))
    return _MANY_NL.sub("\n\n", text).strip()

def hash64(text):
    return int.from_bytes(hashlib.blake2b(text.encode("utf-8", "ignore"), digest_size=8).digest(), "little")

def doc_split(source, idx):
    return "val" if zlib.crc32(f"{source}:{idx}".encode()) % 10000 < VAL_PER_10K else "train"

def list_phase1_files(local_raw_dir=None):
    """Return {'train': {source: repo_path}, 'test': repo_path, 'sizes': {repo_path: bytes}}."""
    if local_raw_dir:
        files = [str(Path(p).relative_to(local_raw_dir)) for p in glob.glob(f"{local_raw_dir}/**/*.json*", recursive=True)]
        sizes = {f: os.path.getsize(Path(local_raw_dir) / f) for f in files}
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        try:
            entries = list(api.list_repo_tree(HF_DATASET_REPO, repo_type="dataset",
                                              path_in_repo="Phase_1_Pretraining", recursive=True))
            files = [e.path for e in entries if getattr(e, "size", None) is not None]
            sizes = {e.path: e.size for e in entries if getattr(e, "size", None) is not None}
        except Exception as e:
            print("list_repo_tree failed (", e, ") -> falling back to list_repo_files")
            files = [f for f in api.list_repo_files(HF_DATASET_REPO, repo_type="dataset") if f.startswith("Phase_1")]
            sizes = {}
    out = {"train": {}, "test": None, "sizes": sizes}
    for f in files:
        name = Path(f).name.lower()
        if not name.endswith((".json", ".jsonl")): continue
        if "test" in name: out["test"] = f
        for s in SOURCE_NAMES:
            if name.startswith("train") and s in name: out["train"][s] = f
    return out

def get_raw_file(repo_path, local_raw_dir=None):
    if local_raw_dir:
        return Path(local_raw_dir) / repo_path
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, repo_path, repo_type="dataset", token=HF_TOKEN,
                                local_dir=str(SCRATCH / "raw")))

def drop_raw_file(path, local_raw_dir=None):
    if local_raw_dir: return          # never delete user-provided files
    try: os.remove(path); print("deleted raw", path)
    except OSError: pass

env=kaggle  ROOT=/kaggle/working/cs781_p1  SCRATCH=/tmp/cs781_scratch
  disk @ /tmp/cs781_scratch: free 1146.0 GB / total 8758.4 GB
  disk @ /kaggle/working/cs781_p1: free 20.9 GB / total 21.0 GB
HF token found: True


## 1. Configuration

In [2]:
import hashlib, torch, torch.nn as nn, torch.nn.functional as F, zipfile, threading, queue as _queue, traceback
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score
from sklearn.linear_model import LogisticRegression
try: display
except NameError: display = print

P5_SMOKE = os.environ.get("CS781_P5_SMOKE") == "1"

P1_SOURCES = {
    "A0": dict(dataset="cs781-p1-model-a0-sc80", run_name="A0_gpt2_110M_SC80_epoch_1000M"),
    "A1": dict(dataset="cs781-p1-model-a1-sc80", run_name="A1_modern_110M_SC80_epoch_1000M"),
    "B":  dict(dataset="cs781-p1-model-b", run_name="B_encdec_6x6_SC80_epoch_1000M"),
    "E":  dict(dataset="data-partial-e-prefixlm-110m-sc80-epoch-1000m", run_name="E_prefixlm_110M_SC80_epoch_1000M"),
    "F":  dict(dataset="cs781-p1-model-f", run_name="F_localglobal_gqa_alibi_SC80_epoch_1000M"),
}

DEFAULTS = dict(
    model="E", attn="bidir", init="pretrained", keep_layers=None,        # keep_layers=k: only the first k blocks (smaller student)
    method="full", lr=None,
    lora_r=16, lora_alpha=32, lora_targets="all", lorma_variant="plus", topk=4, adapter_dim=64, prompt_len=16,
    # ---- document representation ----
    rep="head",            # "head" (first seq_len tokens) | "tail" (last) | "chunks" (n_chunks x chunk_len)
    seq_len=512, n_chunks=4, chunk_len=512,
    chunk_select="spread", # documents with more chunks than n_chunks: "spread" (evenly over the document) | "first" | "last"
    tok_pool="mean",       # "mean" | "last" (end separator — the summary position of a causal model) | "attn"
    chunk_pool="attn",     # linear head + rep="chunks": "attn" | "mean"
    # ---- head / loss ----
    head="lwan",           # "lwan" (label-wise attention over all tokens) | "linear" (pooled document vector)
    lwan_dim=256, head_dropout=0.1, backbone_dropout=0.1,
    loss="asl",            # "bce" | "bce_pos" (positive weight (neg/pos)^pos_weight_power) | "focal" | "asl"
    asl_gamma_neg=4.0, asl_gamma_pos=0.0, asl_clip=0.05, focal_gamma=2.0, pos_weight_power=0.5,
    # ---- data / optimisation ----
    max_train_docs=None, epochs=3, batch=8, accum=2, lr_head=1e-3, weight_decay=0.01, warmup=0.06, grad_clip=1.0,
    teacher=None, kd_alpha=0.5, kd_T=2.0,
    # ---- hierarchical two-stage ----
    hier_from=None, hier_chunks=16, hier_epochs=15, hier_head="lwan",
    seed=0, est_h=None,
)
METHOD_LR = {"full": 3e-5, "frozen": 0.0, "topk": 5e-5, "gradual": 3e-5, "bitfit": 1e-3, "lorma": 5e-4, "lora": 5e-4,
             "rslora": 5e-4, "adapter": 1e-3, "ia3": 3e-3, "prompt": 3e-2}
CAUSAL = dict(attn="causal", tok_pool="last")                  # how the decoder-only models are used (Phase 3/4 lesson)
CH4 = dict(rep="chunks", n_chunks=4, batch=2, accum=8)

EXPERIMENTS = {
    # ---- Stage 1 (E, full FT, all training docs): loss, head, which part of the document ----
    "s1_E_head512_linear_bce":    dict(head="linear", loss="bce", epochs=2),
    "s1_E_head512_linear_bcepos": dict(head="linear", loss="bce_pos", epochs=2),
    "s1_E_head512_linear_focal":  dict(head="linear", loss="focal", epochs=2),
    "s1_E_head512_linear_asl":    dict(head="linear", loss="asl", epochs=2),
    "s1_E_head512_lwan_asl":      dict(),                                         # the default set-up (also E's stage-2 entry)
    "s1_E_tail512_lwan_asl":      dict(rep="tail"),
    "s1_E_chunks4_lwan_asl":      dict(**CH4),
    "s1_E_chunks4_attn_asl":      dict(**CH4, head="linear"),
    # ---- Stage 2: Phase-1 backbones (default set-up: first 512 tokens, label attention, ASL) ----
    "s2_A0":                      dict(model="A0", **CAUSAL),
    "s2_A1":                      dict(model="A1", **CAUSAL),
    "s2_F":                       dict(model="F", **CAUSAL),
    "s2_B":                       dict(model="B"),
    # ---- Stage 3: adaptation techniques on E (+ Phase-1 transfer check) ----
    "s3_E_frozen":                dict(method="frozen"),
    "s3_E_top4":                  dict(method="topk", topk=4),
    "s3_E_gradual":               dict(method="gradual"),
    "s3_E_bitfit":                dict(method="bitfit"),
    "s3_E_lorma":                 dict(method="lorma"),
    "s3_E_adapter":               dict(method="adapter"),
    "s3_E_prompt":                dict(method="prompt"),
    "s3_A1_random_init":          dict(model="A1", **CAUSAL, init="random", lr=3e-4),
    # ---- Stage 4 (attach the stage 1-3 outputs): hierarchical models, distillation, LoRMA on other backbones ----
    "s4_hier16_E":                dict(hier_from="s1_E_head512_lwan_asl"),
    "s4_hier16_E_chunks4":        dict(hier_from="s1_E_chunks4_lwan_asl"),
    "s4_hier16_A0":               dict(hier_from="s2_A0"),
    "s4_hier16_A1":               dict(hier_from="s2_A1"),
    "s4_hier16_F":                dict(hier_from="s2_F"),
    "s4_hier16_E_lorma":          dict(hier_from="s3_E_lorma"),
    "s4_A1_6L_control":           dict(model="A1", **CAUSAL, keep_layers=6),
    "s4_A1_6L_kd":                dict(model="A1", **CAUSAL, keep_layers=6, teacher="s1_E_head512_lwan_asl"),
    "s4_B_kd":                    dict(model="B", teacher="s1_E_head512_lwan_asl"),
    "s4_A0_lorma":                dict(model="A0", **CAUSAL, method="lorma", lr=1e-4),   # A0 + LoRMA diverged at 5e-4 in Phase 4
    "s4_A1_lorma":                dict(model="A1", **CAUSAL, method="lorma"),
    "s4_F_lorma":                 dict(model="F", **CAUSAL, method="lorma"),
}
QUEUES = {f"stage{i}": [k for k in EXPERIMENTS if k.startswith(f"s{i}_")] for i in (1, 2, 3, 4)}
RUN_QUEUE = QUEUES["stage2"]

if P5_SMOKE:
    P1_SOURCES = {k: dict(dataset="", run_name=n) for k, n in {
        "A0": "SMOKE_A0_gpt2_110M_mix50_300M", "A1": "SMOKE_A1_modern_110M_mix50_300M",
        "B": "SMOKE_B_encdec_6x6_mix50_epoch_300M", "E": "SMOKE_E_prefixlm_110M_mix50_epoch_300M",
        "F": "SMOKE_F_localglobal_gqa_alibi_mix50_epoch_300M"}.items()}
    DEFAULTS.update(epochs=1, seq_len=128, chunk_len=128, max_train_docs=400, hier_epochs=2, hier_chunks=3, prompt_len=4)
    for k in EXPERIMENTS:
        EXPERIMENTS[k].update(epochs=1, max_train_docs=400)
        if EXPERIMENTS[k].get("keep_layers"): EXPERIMENTS[k]["keep_layers"] = 1
    RUN_QUEUE = [x for x in os.environ.get("CS781_P5_QUEUE", "s1_E_head512_lwan_asl").split(",") if x]

def exp_cfg(name):
    cfg = dict(DEFAULTS); cfg.update(EXPERIMENTS[name]); cfg["name"] = name
    if cfg["lr"] is None: cfg["lr"] = METHOD_LR[cfg["method"]]
    if cfg["est_h"] is None:                                     # rough T4 hours (used to skip what cannot finish in time)
        cfg["est_h"] = (0.9 + 0.35 * cfg["hier_chunks"] / 16) if cfg["hier_from"] else \
                       (0.45 * cfg["epochs"] * (cfg["n_chunks"] / 1.7 if cfg["rep"] == "chunks" else 1) + (0.15 if cfg["teacher"] else 0))
    return cfg

# longest first within each stage, so that a short run (not a 3-hour one) is what gets skipped if the session runs out of time
for _q in QUEUES.values(): _q.sort(key=lambda n: -exp_cfg(n)["est_h"])
QUEUES["stage4a"] = [n for n in QUEUES["stage4"] if not exp_cfg(n)["hier_from"]]   # KD + LoRMA (~6 h on 2 GPUs)
QUEUES["stage4b"] = [n for n in QUEUES["stage4"] if exp_cfg(n)["hier_from"]]       # hierarchical (~4 h on 2 GPUs)
EXPS_DIR = ROOT / "p5" / "exps"; EXPS_DIR.mkdir(parents=True, exist_ok=True)
NUM_GPUS = torch.cuda.device_count()
print("GPUs:", NUM_GPUS, "| queue:", RUN_QUEUE)

GPUs: 2 | queue: ['s2_A0', 's2_A1', 's2_F', 's2_B']


## 2. Inputs: token files, clean-up, Phase-1 checkpoints
P5_01 found ≈ 1% identical texts across splits (153 test documents also in train, 98 also in dev). Training on, or
tuning thresholds on, labelled copies of test documents would use test labels, so: those train documents are
dropped, and dev is restricted to documents that are not in test ("clean dev"). Nothing is ever trained on dev.

In [3]:
META = load_json(find_artifact("p5/data/meta.json")) if find_artifact("p5/data/meta.json") else None
assert META is not None, "attach the P5_01 output (p5/data/…)"
N_LABELS = int(META["n_labels"]); BUCKET = np.array(META["label_bucket"], dtype=object)

class Split:
    def __init__(self, name):
        self.name = name
        self.tok = np.memmap(find_artifact(f"p5/data/{name}_tokens.bin"), dtype=np.uint16, mode="r")
        off = np.load(find_artifact(f"p5/data/{name}_offsets.npy")); self.start, self.end = off[:-1], off[1:]
        m = load_json(find_artifact(f"p5/data/{name}_meta.json"))
        self.ids = m["ids"]; self.Y = None
        if m["labels"] is not None:
            self.Y = np.zeros((len(self.ids), N_LABELS), np.uint8)
            for i, l in enumerate(m["labels"]): self.Y[i, l] = 1
        self.lengths = self.end - self.start
    def __len__(self): return len(self.ids)
    def tokens(self, i): return np.asarray(self.tok[self.start[i]:self.end[i]], dtype=np.int64)
    def doc_hash(self, i): return hashlib.blake2b(np.asarray(self.tok[self.start[i]:self.end[i]]).tobytes(), digest_size=8).digest()
    def subset(self, keep):
        s = Split.__new__(Split); s.name, s.tok = self.name, self.tok; keep = np.asarray(keep)
        s.start, s.end, s.lengths = self.start[keep], self.end[keep], self.lengths[keep]
        s.ids = [self.ids[i] for i in keep]; s.Y = self.Y[keep] if self.Y is not None else None
        return s

SPLITS = {s: Split(s) for s in ("train", "dev", "test")}
_test_h = {SPLITS["test"].doc_hash(i) for i in range(len(SPLITS["test"]))}
for s in ("train", "dev"):
    dup = np.array([SPLITS[s].doc_hash(i) in _test_h for i in range(len(SPLITS[s]))], bool)
    if dup.any(): SPLITS[s] = SPLITS[s].subset(np.where(~dup)[0])
    print(f"{s}: {dup.sum()} documents identical to a test document removed")
for s, sp in SPLITS.items():
    print(f"{s}: {len(sp)} docs | tokens/doc mean {sp.lengths.mean():.0f}, median {np.median(sp.lengths):.0f}"
          + (f" | statutes/doc {sp.Y.sum(1).mean():.2f}" if sp.Y is not None else ""))
CNT_TRAIN = SPLITS["train"].Y.sum(0).astype(np.float64)
SEP_ID, PAD_ID = 0, 1

INPUT_ROOTS = [os.environ["CS781_P5_INPUTS"]] if os.environ.get("CS781_P5_INPUTS") else (["/kaggle/input"] if IN_KAGGLE else [])
EXP_DIRS = {}
for r in INPUT_ROOTS:
    for f in sorted(glob.glob(f"{r}/**/result.json", recursive=True) + glob.glob(f"{r}/**/trainable.pt", recursive=True)):
        p = Path(f).parent
        if p.name in EXPERIMENTS: EXP_DIRS.setdefault(p.name, []).append(p)
def find_exp_file(name, fname):
    p = EXPS_DIR / name / fname
    if p.exists(): return p
    for d in EXP_DIRS.get(name, []):
        if (d / fname).exists(): return d / fname
    return None
if INPUT_ROOTS:
    n = 0
    for name, dirs in EXP_DIRS.items():
        for dd in dirs:
            for src in dd.glob("*"):
                if src.is_file() and src.suffix in (".json", ".zip", ".txt") and src.name != "error.txt":
                    dst = EXPS_DIR / name / src.name
                    if not dst.exists(): dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); n += 1
    print("attached earlier experiments:", len(EXP_DIRS), "| with weights:", sorted(k for k in EXP_DIRS if find_exp_file(k, "trainable.pt")))
    print("restored", n, "result files")

def find_p1_dir(key):
    src = P1_SOURCES[key]
    roots = [os.environ["CS781_P1_RUNS"]] if os.environ.get("CS781_P1_RUNS") else []
    roots += [str(ROOT / "runs")] + (["/kaggle/input"] if IN_KAGGLE else [])
    cands = sorted({c for r in roots for c in glob.glob(f"{r}/**/weights_final.pt", recursive=True)})
    by_run = [c for c in cands if Path(c).parent.name == src["run_name"]]
    by_ds = [c for c in cands if src["dataset"] and src["dataset"].lower() in [p.lower() for p in Path(c).parts]]
    hit = (by_run or by_ds or [None])[0]
    return Path(hit).parent if hit else None
P1_DIRS = {k: find_p1_dir(k) for k in P1_SOURCES}
for k, p in P1_DIRS.items(): print(f"  {k}: {p if p else 'NOT FOUND'}")

train: 108 documents identical to a test document removed
dev: 96 documents identical to a test document removed
train: 42207 docs | tokens/doc mean 1707, median 646 | statutes/doc 3.70
dev: 10085 docs | tokens/doc mean 1916, median 815 | statutes/doc 3.91
test: 13019 docs | tokens/doc mean 1851, median 797
attached earlier experiments: 0 | with weights: []
restored 0 result files
  A0: /kaggle/input/datasets/aarinsheth/cs781-p1-model-a0-sc80/cs781_p1/runs/A0_gpt2_110M_SC80_epoch_1000M
  A1: /kaggle/input/datasets/aarinsheth/cs781-p1-model-a1-sc80/cs781_p1/runs/A1_modern_110M_SC80_epoch_1000M
  B: /kaggle/input/datasets/aaravoswal/cs781-p1-model-b/cs781_p1/runs/B_encdec_6x6_SC80_epoch_1000M
  E: /kaggle/input/datasets/chrishitha2403/data-partial-e-prefixlm-110m-sc80-epoch-1000m/cs781_p1/runs/E_prefixlm_110M_SC80_epoch_1000M
  F: /kaggle/input/datasets/chrishitha2403/cs781-p1-model-f/cs781_p1/runs/F_localglobal_gqa_alibi_SC80_epoch_1000M


## 3. Model classes (NB3) and backbone + adaptation methods (P3_02)

In [4]:
# (model classes copied unchanged from NB3's model cell)
class CausalSelfAttention(nn.Module):
    def __init__(self, d, h, dropout):
        super().__init__()
        self.h, self.hd = h, d // h
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.proj.RESIDUAL_SCALE = True
        self.dropout = dropout

    def forward(self, x, past=None, mask=None, use_cache=False):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.h, self.hd).transpose(1, 2) for t in (q, k, v)]
        if past is not None:
            k = torch.cat([past[0], k], dim=2); v = torch.cat([past[1], v], dim=2)
        present = (k, v) if use_cache else None
        drop = self.dropout if self.training else 0.0
        if mask is None:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=drop)
        else:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=drop)
        return self.proj(y.transpose(1, 2).reshape(B, T, C)), present

class MLP(nn.Module):
    def __init__(self, d, dropout):
        super().__init__()
        self.fc, self.proj = nn.Linear(d, 4 * d), nn.Linear(4 * d, d)
        self.proj.RESIDUAL_SCALE = True
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x), approximate="tanh")))

class Block(nn.Module):
    def __init__(self, d, h, dropout):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn, self.mlp = CausalSelfAttention(d, h, dropout), MLP(d, dropout)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, past=None, mask=None, use_cache=False):
        a, present = self.attn(self.ln1(x), past, mask, use_cache)
        x = x + self.drop(a)
        x = x + self.mlp(self.ln2(x))
        return x, present

class GPT2LM(nn.Module):
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, **_):
        super().__init__()
        self.ctx_len = ctx_len
        self.wte, self.wpe = nn.Embedding(vocab, d_model), nn.Embedding(ctx_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False)
        self.lm_head.weight = self.wte.weight                     # weight tying
        for name, m in self.named_modules():
            if isinstance(m, nn.Linear):
                std = 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02
                nn.init.normal_(m.weight, 0.0, std)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        x = self.drop(self.wte(idx) + self.wpe(pos))
        new_cache = []
        for i, blk in enumerate(self.blocks):
            x, present = blk(x, cache[i] if cache is not None else None, mask, use_cache)
            new_cache.append(present)
        x = self.ln_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new_cache if use_cache else None)

# ------------------------------------------------------------------ A1: modern decoder
class RMSNorm(nn.Module):
    """x / sqrt(mean(x²)) · g — no mean subtraction, no bias (cheaper than LayerNorm, equally stable). Computed in fp32."""
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight).to(x.dtype)

def rope_tables(head_dim, n_pos, base):
    inv = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    ang = torch.outer(torch.arange(n_pos).float(), inv)                    # (n_pos, head_dim/2)
    return torch.cos(ang), torch.sin(ang)

def apply_rope(x, cos, sin):
    """Rotate each (x1_i, x2_i) pair of the head dimension by angle pos·θ_i. The dot product q·k then depends only on
    the RELATIVE distance between the two positions. x: (B, H, T, D); cos/sin: (B or 1, 1, T, D/2)."""
    d = x.size(-1) // 2
    x1, x2 = x[..., :d].float(), x[..., d:].float()
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).to(x.dtype)

class ModernAttention(nn.Module):
    def __init__(self, d, h, kv, dropout, rope, causal=True):
        super().__init__()
        self.h, self.kv, self.hd, self.rope, self.dropout, self.causal = h, kv, d // h, rope, dropout, causal
        self.q = nn.Linear(d, d, bias=False)
        self.kvp = nn.Linear(d, 2 * kv * self.hd, bias=False)             # GQA when kv < h: fewer key/value heads
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
    def forward(self, x, cos, sin, past=None, mask=None, use_cache=False):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k, v = self.kvp(x).view(B, T, 2, self.kv, self.hd).permute(2, 0, 3, 1, 4)
        if self.rope:
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if past is not None:
            k = torch.cat([past[0], k], 2); v = torch.cat([past[1], v], 2)
        present = (k, v) if use_cache else None
        if self.kv != self.h:
            k, v = k.repeat_interleave(self.h // self.kv, 1), v.repeat_interleave(self.h // self.kv, 1)
        drop = self.dropout if self.training else 0.0
        # no explicit mask: causal for decoders (but a single cached query may see every past key), full for encoders
        y = (F.scaled_dot_product_attention(q, k, v, is_causal=self.causal and past is None, dropout_p=drop)
             if mask is None else F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=drop))
        return self.o(y.transpose(1, 2).reshape(B, T, C)), present

class SwiGLU(nn.Module):
    """out = W2( SiLU(W1 x) ⊙ W3 x ): a learned gate decides, per hidden unit, how much of W3 x passes through."""
    def __init__(self, d, hidden, dropout):
        super().__init__()
        self.w1, self.w3 = nn.Linear(d, hidden, bias=False), nn.Linear(d, hidden, bias=False)
        self.w2 = nn.Linear(hidden, d, bias=False); self.w2.RESIDUAL_SCALE = True
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.w2(F.silu(self.w1(x)) * self.w3(x)))

class GeluMLP(nn.Module):
    def __init__(self, d, hidden, dropout):
        super().__init__()
        self.fc, self.proj = nn.Linear(d, hidden, bias=False), nn.Linear(hidden, d, bias=False)
        self.proj.RESIDUAL_SCALE = True; self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x), approximate="tanh")))

class ModernBlock(nn.Module):
    def __init__(self, d, h, kv, hidden, dropout, norm, mlp, rope):
        super().__init__()
        Norm = RMSNorm if norm == "rms" else nn.LayerNorm
        self.n1, self.n2 = Norm(d), Norm(d)
        self.attn = ModernAttention(d, h, kv, dropout, rope)
        self.mlp = SwiGLU(d, hidden, dropout) if mlp == "swiglu" else GeluMLP(d, hidden, dropout)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, cos, sin, past=None, mask=None, use_cache=False):
        a, present = self.attn(self.n1(x), cos, sin, past, mask, use_cache)
        x = x + self.drop(a)
        return x + self.mlp(self.n2(x)), present

class ModernLM(nn.Module):
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, n_kv_head=None, ffn_hidden=2048,
                 rope_base=10000, pos="rope", norm="rms", mlp="swiglu", **_):
        super().__init__()
        self.ctx_len, self.use_rope = ctx_len, pos == "rope"
        self.wte = nn.Embedding(vocab, d_model)
        self.wpe = None if self.use_rope else nn.Embedding(ctx_len, d_model)
        hd = d_model // n_head
        cos, sin = rope_tables(hd, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([ModernBlock(d_model, n_head, n_kv_head or n_head, ffn_hidden, dropout, norm, mlp,
                                                 self.use_rope) for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model) if norm == "rms" else nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False)
        self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        x = self.wte(idx)
        if self.wpe is not None: x = x + self.wpe(pos)
        cos, sin = self.rope_cos[pos][:, None], self.rope_sin[pos][:, None]
        x = self.drop(x)
        new_cache = []
        for i, blk in enumerate(self.blocks):
            x, present = blk(x, cos, sin, cache[i] if cache is not None else None, mask, use_cache)
            new_cache.append(present)
        x = self.norm_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new_cache if use_cache else None)

# ------------------------------------------------------------------ B: encoder-decoder
class CrossAttention(nn.Module):
    """decoder queries attend to the encoder's output ("memory"); keys/values are computed once per prompt"""
    def __init__(self, d, h):
        super().__init__()
        self.h, self.hd = h, d // h
        self.q = nn.Linear(d, d, bias=False); self.kvp = nn.Linear(d, 2 * d, bias=False)
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
    def forward(self, x, mem, mem_valid=None, kv=None):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        if kv is None:
            S = mem.size(1)
            kv = tuple(self.kvp(mem).view(B, S, 2, self.h, self.hd).permute(2, 0, 3, 1, 4))
        k, v = kv
        mask = None if mem_valid is None else mem_valid[:, None, None, :]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.o(y.transpose(1, 2).reshape(B, T, C)), kv

class EncoderBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.attn = ModernAttention(d, h, h, dropout, rope=True, causal=False)     # bidirectional
        self.mlp = SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, mask=None):
        x = x + self.attn(self.n1(x), cos, sin, None, mask, False)[0]
        return x + self.mlp(self.n2(x))

class DecoderBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2, self.n3 = RMSNorm(d), RMSNorm(d), RMSNorm(d)
        self.self_attn = ModernAttention(d, h, h, dropout, rope=True, causal=True)
        self.cross = CrossAttention(d, h)
        self.mlp = SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, mem, mem_valid, cache=None, use_cache=False):
        a, self_kv = self.self_attn(self.n1(x), cos, sin, cache["self"] if cache else None, None, use_cache)
        x = x + a
        c, cross_kv = self.cross(self.n2(x), mem, mem_valid, cache["cross"] if cache else None)
        x = x + c
        return x + self.mlp(self.n3(x)), ({"self": self_kv, "cross": cross_kv} if use_cache else None)

class EncDecRunner:
    """generation for the encoder-decoder: encode the prompt once, then decode from <|bos|> one token at a time"""
    def __init__(self, model, rows, pad_id, device, amp):
        self.model, self.device, self.amp = model, device, amp
        B, L = len(rows), max(len(r) for r in rows)
        idx = torch.full((B, L), pad_id, dtype=torch.long); valid = torch.zeros((B, L), dtype=torch.bool)
        for r, ids_ in enumerate(rows):                                        # right padding (bidirectional encoder)
            idx[r, :len(ids_)] = torch.tensor(ids_); valid[r, :len(ids_)] = True
        self.idx, self.mem_valid = idx.to(device), valid.to(device)
    def prefill(self):
        with self.amp:
            self.mem = self.model.encode(self.idx, self.mem_valid)
            bos = torch.full((self.idx.size(0), 1), self.model.bos_id, dtype=torch.long, device=self.device)
            logits, self.cache = self.model.decode(bos, self.mem, self.mem_valid, 0, None, use_cache=True)
        self.t = 0
        return logits[:, -1].float()
    def step(self, nxt):
        self.t += 1
        with self.amp:
            logits, self.cache = self.model.decode(nxt[:, None], self.mem, self.mem_valid, self.t, self.cache, use_cache=True)
        return logits[:, -1].float()
    def reorder(self, idx):
        self.mem, self.mem_valid = self.mem[idx], self.mem_valid[idx]; self.cache = reorder_cache(self.cache, idx)

class EncDecLM(nn.Module):
    """B: Transformer encoder-decoder. Training: each window is split at a random point E — the encoder reads tokens
    [0, E) bidirectionally, the decoder (starting from <|bos|>, cross-attending to the encoder) predicts tokens [E, T].
    At test time the prompt goes to the encoder and the continuation is decoded — exactly the task's shape."""
    is_encdec = True
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, ffn_hidden=2048, rope_base=10000,
                 enc_layers=None, enc_len_eval=768, bos_id=2, **_):
        super().__init__()
        ne = enc_layers or n_layer // 2; nd = n_layer - ne
        self.bos_id, self.enc_len_eval, self.ctx_len = bos_id, enc_len_eval, ctx_len
        self.wte = nn.Embedding(vocab, d_model)
        cos, sin = rope_tables(d_model // n_head, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.enc = nn.ModuleList([EncoderBlock(d_model, n_head, ffn_hidden, dropout) for _ in range(ne)])
        self.dec = nn.ModuleList([DecoderBlock(d_model, n_head, ffn_hidden, dropout) for _ in range(nd)])
        self.enc_norm, self.norm_f = RMSNorm(d_model), RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False); self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)
    def _rope(self, start, T, device):
        p = torch.arange(start, start + T, device=device)[None]
        return self.rope_cos[p][:, None], self.rope_sin[p][:, None]
    def encode(self, idx, valid=None):
        cos, sin = self._rope(0, idx.size(1), idx.device)
        x = self.wte(idx); mask = None if valid is None else valid[:, None, None, :]
        for blk in self.enc: x = blk(x, cos, sin, mask)
        return self.enc_norm(x)
    def decode(self, idx, mem, mem_valid, start, cache=None, use_cache=False):
        cos, sin = self._rope(start, idx.size(1), idx.device)
        x = self.wte(idx); new = []
        for i, blk in enumerate(self.dec):
            x, c = blk(x, cos, sin, mem, mem_valid, cache[i] if cache else None, use_cache); new.append(c)
        return self.lm_head(self.norm_f(x)), (new if use_cache else None)
    def forward(self, idx, targets=None, enc_len=None, **_):
        B, T = idx.shape
        E = int(enc_len or min(self.enc_len_eval, T - 1))
        mem = self.encode(idx[:, :E])
        dec_in = torch.cat([torch.full((B, 1), self.bos_id, dtype=torch.long, device=idx.device), idx[:, E:]], 1)
        logits, _ = self.decode(dec_in, mem, None, 0)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets[:, E - 1:].reshape(-1))
        return logits, loss, None
    def make_runner(self, rows, pad_id, device, amp):
        return EncDecRunner(self, rows, pad_id, device, amp)

# ------------------------------------------------------------------ C: retention network (RetNet-style), D: hybrid
class Retention(nn.Module):
    """Multi-scale retention (Sun et al. 2023, "Retentive Network"). Attention without softmax:
         parallel form (training):   O = (Q Kᵀ ⊙ D) V,   D[n,m] = γ^(n−m) for n ≥ m, else 0
         recurrent form (generation): S_n = γ·S_{n−1} + k_nᵀ v_n,   o_n = q_n S_n
       Both give identical outputs. Each head has its own decay γ (short- to long-range memory), so generation needs
       only a fixed-size state per head — no growing KV-cache, O(1) cost per new token.
       Per-head normalisation + a SiLU output gate as in RetNet; RoPE on q/k for relative position."""
    def __init__(self, d, h):
        super().__init__()
        self.h, self.hd = h, d // h
        self.q, self.k = nn.Linear(d, d, bias=False), nn.Linear(d, d, bias=False)
        self.v, self.g = nn.Linear(d, d, bias=False), nn.Linear(d, d, bias=False)
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
        self.register_buffer("log_gamma", torch.log(1 - 2.0 ** (-5 - torch.arange(h, dtype=torch.float32))), persistent=False)
    def forward(self, x, cos, sin, state=None, valid=None, use_cache=False):
        B, T, C = x.shape
        q, k, v = [m(x).view(B, T, self.h, self.hd).transpose(1, 2) for m in (self.q, self.k, self.v)]
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if valid is not None:                                   # left padding: padded positions contribute nothing
            vm = valid[:, None, :, None].to(k.dtype); k, v = k * vm, v * vm
        q = q * self.hd ** -0.5
        lg = self.log_gamma.view(1, self.h, 1, 1)
        new_state = None
        if state is None:                                       # parallel form
            n = torch.arange(T, device=x.device, dtype=torch.float32)
            diff = n[:, None] - n[None, :]
            D = torch.where(diff >= 0, torch.exp(diff * lg), torch.zeros((), device=x.device)).to(q.dtype)
            o = ((q @ k.transpose(-1, -2)) * D) @ v
            if use_cache:
                w = torch.exp((T - 1 - n).view(1, 1, T, 1) * lg)                       # γ^(T−1−m)
                new_state = (k.float() * w).transpose(-1, -2) @ v.float()            # (B, H, hd, hd)
        else:                                                   # recurrent form, one token at a time
            new_state = state * torch.exp(lg) + k.float().transpose(-1, -2) @ v.float()
            o = (q.float() @ new_state).to(q.dtype)
        o = F.layer_norm(o.float(), (self.hd,)).to(x.dtype)     # per-head normalisation (GroupNorm, no affine)
        o = o.transpose(1, 2).reshape(B, T, C)
        return self.o(F.silu(self.g(x)) * o), (new_state if use_cache else None)

class RetBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.ret, self.mlp = Retention(d, h), SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, state=None, valid=None, use_cache=False):
        r, s = self.ret(self.n1(x), cos, sin, state, valid, use_cache)
        x = x + r
        return x + self.mlp(self.n2(x)), s

class RetNetLM(nn.Module):
    """C (attn_every=0): every block is retention.  D (attn_every=k): every k-th block is full softmax attention
    (ModernBlock), the rest retention — retention supplies cheap long-range decayed memory, the few attention layers
    give exact recall of specific earlier tokens (names, section numbers)."""
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, ffn_hidden=1792, rope_base=10000,
                 attn_every=0, n_kv_head=None, **_):
        super().__init__()
        self.ctx_len = ctx_len
        self.wte = nn.Embedding(vocab, d_model)
        cos, sin = rope_tables(d_model // n_head, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.kinds = ["attn" if attn_every and (i + 1) % attn_every == 0 else "ret" for i in range(n_layer)]
        self.blocks = nn.ModuleList([
            ModernBlock(d_model, n_head, n_kv_head or n_head, ffn_hidden, dropout, "rms", "swiglu", True) if kd == "attn"
            else RetBlock(d_model, n_head, ffn_hidden, dropout) for kd in self.kinds])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False); self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        cos, sin = self.rope_cos[pos][:, None], self.rope_sin[pos][:, None]
        valid = mask[:, 0, -1, :] if (mask is not None and cache is None) else None   # key validity at prefill
        x = self.wte(idx); new = []
        for i, (kd, blk) in enumerate(zip(self.kinds, self.blocks)):
            c = cache[i] if cache is not None else None
            if kd == "attn": x, c = blk(x, cos, sin, c, mask, use_cache)
            else:            x, c = blk(x, cos, sin, c, valid, use_cache)
            new.append(c)
        x = self.norm_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new if use_cache else None)

# ------------------------------------------------------------------ E: prefix-LM (encoder/hybrid adapted for LM)
class PrefixLM(ModernLM):
    """E: the A1 stack with a different ATTENTION MASK. Prompt (prefix) tokens attend to the whole prompt in both
    directions, like a BERT encoder; continuation tokens attend causally, like GPT. One shared stack plays both roles.
    Training: split each window at a random E, loss only on the continuation (predictions from position E−1 on).
    Generation: prefill the prompt with the bidirectional mask, then ordinary cached causal decoding."""
    def __init__(self, *a, enc_len_eval=768, **kw):
        super().__init__(*a, **kw); self.enc_len_eval = enc_len_eval
    def forward(self, idx, targets=None, enc_len=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        if targets is None:                                       # generation path (mask supplied by the runner)
            return super().forward(idx, None, pos, mask, cache, use_cache, last_only)
        T = idx.size(1); E = int(enc_len or min(self.enc_len_eval, T - 1))
        i = torch.arange(T, device=idx.device)
        allowed = (i[None, :] <= i[:, None]) | ((i[:, None] < E) & (i[None, :] < E))   # causal ∪ (prefix × prefix)
        logits, _, _ = super().forward(idx, None, None, allowed[None, None], None, False, False)
        loss = F.cross_entropy(logits[:, E - 1:].float().reshape(-1, logits.size(-1)), targets[:, E - 1:].reshape(-1))
        return logits, loss, None
    def make_runner(self, rows, pad_id, device, amp):
        return PrefixRunner(self, rows, pad_id, device, amp)

# ------------------------------------------------------------------ F: efficient local-global decoder
def alibi_slopes(n):
    """ALiBi head slopes (Press et al. 2022): geometric sequence; extra heads interleaved when n is not a power of 2"""
    def pow2(m): return [2 ** (-8 * (k + 1) / m) for k in range(m)]
    if math.log2(n).is_integer(): return pow2(n)
    p = 2 ** math.floor(math.log2(n))
    return pow2(p) + pow2(2 * p)[0::2][: n - p]

class LocalGlobalAttention(nn.Module):
    """Grouped-query attention (h query heads share kv key/value heads) with an ALiBi distance penalty added to the
    attention logits (no position embeddings at all). window=W makes it sliding-window attention: a query only sees
    the previous W tokens; window=None is full causal attention."""
    def __init__(self, d, h, kv, window):
        super().__init__()
        self.h, self.kv, self.hd, self.window = h, kv, d // h, window
        self.q = nn.Linear(d, d, bias=False); self.kvp = nn.Linear(d, 2 * kv * self.hd, bias=False)
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
    def forward(self, x, bias, past=None, use_cache=False):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k, v = self.kvp(x).view(B, T, 2, self.kv, self.hd).permute(2, 0, 3, 1, 4)
        if past is not None:
            k = torch.cat([past[0], k], 2); v = torch.cat([past[1], v], 2)
        present = (k, v) if use_cache else None
        if self.kv != self.h:
            k, v = k.repeat_interleave(self.h // self.kv, 1), v.repeat_interleave(self.h // self.kv, 1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=bias.to(q.dtype))
        return self.o(y.transpose(1, 2).reshape(B, T, C)), present

class EffBlock(nn.Module):
    def __init__(self, d, h, kv, hidden, dropout, window):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.attn, self.mlp = LocalGlobalAttention(d, h, kv, window), SwiGLU(d, hidden, dropout)
    def forward(self, x, bias, past=None, use_cache=False):
        a, present = self.attn(self.n1(x), bias, past, use_cache)
        x = x + a
        return x + self.mlp(self.n2(x)), present

class EfficientLM(nn.Module):
    """F: lightweight decoder. Sliding-window attention (window W) in most layers, full causal attention in every
    k-th layer, grouped-query attention (fewer key/value heads → smaller KV-cache), ALiBi instead of position
    embeddings. Local layers model nearby text; the few global layers can still reach any earlier token."""
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, n_kv_head=4, ffn_hidden=2304,
                 window=256, global_every=4, **_):
        super().__init__()
        self.ctx_len = ctx_len
        self.wte = nn.Embedding(vocab, d_model)
        self.windows = [None if global_every and (i + 1) % global_every == 0 else window for i in range(n_layer)]
        self.blocks = nn.ModuleList([EffBlock(d_model, n_head, n_kv_head, ffn_hidden, dropout, w) for w in self.windows])
        self.register_buffer("slopes", torch.tensor(alibi_slopes(n_head)), persistent=False)
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False); self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)
    def _bias(self, qpos, kpos, key_ok, window):
        """(B|1, H, Tq, Tk) additive mask: −slope·distance for allowed pairs, −inf elsewhere"""
        dist = (qpos[:, :, None] - kpos[:, None, :]).float()               # (B|1, Tq, Tk)
        ok = (dist >= 0) & key_ok[:, None, :]
        if window is not None: ok = ok & (dist < window)
        b = -self.slopes.view(1, -1, 1, 1) * dist[:, None]
        return b.masked_fill(~ok[:, None], float("-inf"))
    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if mask is None:                                          # training / plain forward: positions 0..T-1
            qpos = kpos = torch.arange(T, device=idx.device)[None]
            key_ok = torch.ones((1, T), dtype=torch.bool, device=idx.device)
        else:                                                     # generation: derive key positions from the mask
            key_ok = mask[:, 0, -1, :]                            # validity of every key (past + current)
            kpos = key_ok.long().cumsum(-1) - 1
            qpos = pos
        biases = {w: self._bias(qpos, kpos, key_ok, w) for w in set(self.windows)}
        x = self.wte(idx); new = []
        for i, blk in enumerate(self.blocks):
            x, c = blk(x, biases[self.windows[i]], cache[i] if cache is not None else None, use_cache); new.append(c)
        x = self.norm_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new if use_cache else None)

MODEL_REGISTRY = {"gpt2": GPT2LM, "modern": ModernLM, "encdec": EncDecLM, "retnet": RetNetLM, "hybrid": RetNetLM,
                  "prefixlm": PrefixLM, "efficient": EfficientLM}

def build_model(cfg, vocab):
    return MODEL_REGISTRY[cfg["arch"]](vocab=vocab, **cfg)

def count_params(m):
    return sum(p.numel() for p in m.parameters())      # tied weights counted once

In [5]:
# (backbone + adaptation methods copied unchanged from P3_02)
class Backbone(nn.Module):
    def __init__(self, lm, arch, attn):
        super().__init__()
        self.lm, self.arch = lm, arch
        self.bidir = attn == "bidir" or arch == "encdec"
        self.d = lm.wte.weight.shape[1]
        if arch == "encdec": lm.dec = nn.ModuleList()
        self.ctx_limit = lm.ctx_len if arch == "gpt2" else (2 * lm.ctx_len if arch in ("modern", "prefixlm", "encdec") else 10 ** 9)
    def blocks(self):
        return self.lm.enc if self.arch == "encdec" else self.lm.blocks
    def forward(self, idx, valid, prefix=None):
        lm, (B, T), dev = self.lm, idx.shape, idx.device
        P = 0 if prefix is None else prefix.size(0)
        x = lm.wte(idx)
        if P:
            x = torch.cat([prefix[None].expand(B, -1, -1).to(x.dtype), x], 1)
            valid = torch.cat([torch.ones(B, P, dtype=torch.bool, device=dev), valid], 1)
        L = T + P
        assert L <= self.ctx_limit, f"window {L} longer than the model's position range {self.ctx_limit}"
        pos = torch.arange(L, device=dev)[None]
        mask = valid[:, None, None, :] if self.bidir else None
        if self.arch == "encdec":
            cos, sin = lm._rope(0, L, dev)
            for blk in lm.enc: x = blk(x, cos, sin, mask)
            h = lm.enc_norm(x)
        elif self.arch == "gpt2":
            x = lm.drop(x + lm.wpe(pos))
            for blk in lm.blocks: x, _ = blk(x, None, mask, False)
            h = lm.ln_f(x)
        elif self.arch in ("modern", "prefixlm"):
            if lm.wpe is not None: x = x + lm.wpe(pos)
            cos, sin = lm.rope_cos[pos][:, None], lm.rope_sin[pos][:, None]
            x = lm.drop(x)
            for blk in lm.blocks: x, _ = blk(x, cos, sin, None, mask, False)
            h = lm.norm_f(x)
        elif self.arch == "efficient":
            dist = (pos[:, :, None] - pos[:, None, :]).float()
            biases = {}
            for w in set(lm.windows):
                if self.bidir:
                    ok = valid[:, None, :] & ((dist.abs() < w) if w is not None else torch.ones_like(dist, dtype=torch.bool))
                    ok = ok | torch.eye(L, dtype=torch.bool, device=dev)[None]
                    b = -lm.slopes.view(1, -1, 1, 1) * dist.abs()[:, None]
                else:
                    ok = dist >= 0
                    if w is not None: ok = ok & (dist < w)
                    b = -lm.slopes.view(1, -1, 1, 1) * dist[:, None]
                biases[w] = b.masked_fill(~ok[:, None], float("-inf"))
            for i, blk in enumerate(lm.blocks): x, _ = blk(x, biases[lm.windows[i]], None, False)
            h = lm.norm_f(x)
        else:
            raise ValueError(self.arch)
        return h[:, P:]

class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, rs=False):
        super().__init__()
        self.base, self.scale = base, alpha / (math.sqrt(r) if rs else r)
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

_PERM_IDX, _PERM_LOCK = {}, threading.Lock()
def perm_index(d, device):
    key = (d, str(device))
    with _PERM_LOCK:
        if key not in _PERM_IDX:
            i = torch.arange(d, device=device); _PERM_IDX[key] = (i[None, :] - i[:, None]) % d
        return _PERM_IDX[key]

class LoRMALinear(nn.Module):
    def __init__(self, base, r, alpha, variant="plus"):
        super().__init__()
        d = base.out_features; self.base, self.variant = base, variant
        if variant == "plus":
            self.scale = alpha / r
            self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
            self.B = nn.Parameter(torch.zeros(d, r))
        else:
            A = torch.zeros(r, d); A[0, 0] = 1.0
            B = torch.empty(d, r); nn.init.kaiming_uniform_(B, a=math.sqrt(5)); B[:, 0] = 1.0
            self.A, self.B = nn.Parameter(A), nn.Parameter(B)
    def forward(self, x):
        y = self.base(x)
        if self.variant == "plus":
            return y + self.scale * F.linear(F.linear(y, self.A), self.B)
        M = self.B @ self.A
        return F.linear(y, M.gather(1, perm_index(M.size(0), M.device)))

class IA3Linear(nn.Module):
    """(IA)^3: mode "out" rescales the layer's outputs (keys/values; `mask` limits it to part of a fused q|k|v
    output), mode "in" rescales its inputs (= the FFN hidden activation when wrapped around the down-projection)"""
    def __init__(self, base, mode, mask=None):
        super().__init__()
        self.base, self.mode = base, mode
        n = base.out_features if mode == "out" else base.in_features
        self.l = nn.Parameter(torch.ones(n))
        self.register_buffer("mask", mask if mask is not None else torch.ones(n, dtype=torch.bool), persistent=False)
    def forward(self, x):
        if self.mode == "in": return self.base(x * self.l.to(x.dtype))
        s = torch.where(self.mask, self.l, torch.ones_like(self.l))
        return self.base(x) * s.to(x.dtype)

class BiasLinear(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base; self.bias = nn.Parameter(torch.zeros(base.out_features))
    def forward(self, x): return self.base(x) + self.bias

class AdapterLinear(nn.Module):
    def __init__(self, base, dim):
        super().__init__()
        d = base.out_features; self.base = base
        self.down, self.up = nn.Linear(d, dim), nn.Linear(dim, d)
        nn.init.normal_(self.down.weight, 0, 0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        y = self.base(x); return y + self.up(F.gelu(self.down(y)))

ATTN_TYPES = (CausalSelfAttention, ModernAttention, LocalGlobalAttention)
MLP_TYPES = (MLP, SwiGLU, GeluMLP)

def target_linears(backbone, which):
    out = []
    for parent in list(backbone.blocks().modules()):
        kind = "attn" if isinstance(parent, ATTN_TYPES) else "mlp" if isinstance(parent, MLP_TYPES) else None
        if kind is None or (which == "attn" and kind != "attn"): continue
        for name, child in list(parent.named_children()):
            if isinstance(child, nn.Linear):
                out.append((parent, name, child, bool(getattr(child, "RESIDUAL_SCALE", False))))
    return out

def apply_method(backbone, cfg):
    m = cfg["method"]
    for p in backbone.parameters(): p.requires_grad = (m == "full")
    if m in ("full", "frozen", "prompt"): return
    if m == "topk":
        for blk in list(backbone.blocks())[-cfg["topk"]:]:
            for p in blk.parameters(): p.requires_grad = True
        return
    if m == "ia3":
        for parent in list(backbone.blocks().modules()):
            if isinstance(parent, CausalSelfAttention):                      # fused q|k|v: rescale only k and v
                d = parent.qkv.in_features
                parent.qkv = IA3Linear(parent.qkv, "out", torch.arange(3 * d) >= d)
            elif isinstance(parent, (ModernAttention, LocalGlobalAttention)):
                parent.kvp = IA3Linear(parent.kvp, "out")
            elif isinstance(parent, SwiGLU): parent.w2 = IA3Linear(parent.w2, "in")
            elif isinstance(parent, (MLP, GeluMLP)): parent.proj = IA3Linear(parent.proj, "in")
        return
    which = cfg["lora_targets"] if m in ("lora", "rslora", "lorma") else "all"
    for parent, name, lin, is_out in target_linears(backbone, which):
        if m in ("lora", "rslora"): setattr(parent, name, LoRALinear(lin, cfg["lora_r"], cfg["lora_alpha"], rs=(m == "rslora")))
        elif m == "lorma": setattr(parent, name, LoRMALinear(lin, cfg["lora_r"], cfg["lora_alpha"], cfg["lorma_variant"]))
        elif m == "adapter":
            if is_out: setattr(parent, name, AdapterLinear(lin, cfg["adapter_dim"]))
        elif m == "bitfit":
            if lin.bias is not None: lin.bias.requires_grad = True
            else: setattr(parent, name, BiasLinear(lin))
    if m == "bitfit":
        for mod in backbone.modules():
            if isinstance(mod, (nn.LayerNorm, RMSNorm)):
                for p in mod.parameters(): p.requires_grad = True

def set_backbone_dropout(backbone, p):
    for mod in backbone.modules():
        if isinstance(mod, nn.Dropout): mod.p = p
        if isinstance(mod, (CausalSelfAttention, ModernAttention)): mod.dropout = p

_P1_CACHE, _P1_LOCK = {}, threading.Lock()
def load_phase1(key):
    with _P1_LOCK:
        if key not in _P1_CACHE:
            d = P1_DIRS[key]
            info = load_json(d / "run_info.json") or load_json(d / "summary.json") or {}
            pcfg = dict(info.get("cfg") or {}); assert pcfg, f"no run_info.json/summary.json next to {d}"
            sd = torch.load(d / "weights_final.pt", map_location="cpu", weights_only=False)
            _P1_CACHE[key] = (pcfg, {k: v.float() if v.is_floating_point() else v for k, v in sd.items()})
        return _P1_CACHE[key]

### Additions (as in Phase 4)
* **Gradual unfreezing** (Howard & Ruder 2018): start with only the top block trainable, open one more block at equal
  intervals until all are open at 60% of training.
* **`keep_layers=k`**: keep only the first k blocks — a genuinely smaller student for distillation.
* **Prompt tuning** (Lester et al. 2021): backbone frozen; `prompt_len` trainable vectors prepended to every row.

In [6]:
def apply_method_p5(backbone, cfg):
    if cfg["method"] == "gradual":
        for p in backbone.parameters(): p.requires_grad = False
        return
    apply_method(backbone, cfg)

def gradual_unfreeze(backbone, n_open):
    blocks = list(backbone.blocks())
    for b_i, blk in enumerate(blocks):
        want = b_i >= len(blocks) - n_open
        for p in blk.parameters(): p.requires_grad = want

_P1_CACHE, _P1_LOCK = {}, threading.Lock()
def load_phase1(key):
    with _P1_LOCK:
        if key not in _P1_CACHE:
            d = P1_DIRS[key]; assert d is not None, f"Phase-1 weights for {key} not found"
            info = load_json(d / "run_info.json") or load_json(d / "summary.json") or {}
            pcfg = dict(info.get("cfg") or {}); assert pcfg
            sd = torch.load(d / "weights_final.pt", map_location="cpu", weights_only=False)
            _P1_CACHE[key] = (pcfg, {k: v.float() if v.is_floating_point() else v for k, v in sd.items()})
        return _P1_CACHE[key]

def build_backbone(cfg):
    pcfg, sd = load_phase1(cfg["model"])
    torch.manual_seed(cfg["seed"])
    lm = MODEL_REGISTRY[pcfg["arch"]](vocab=sd["wte.weight"].shape[0], **{k: v for k, v in pcfg.items() if k != "vocab"})
    if cfg["init"] == "pretrained": lm.load_state_dict(sd)
    bb = Backbone(lm, pcfg["arch"], cfg["attn"])
    k = cfg.get("keep_layers")
    if k:
        if bb.arch == "encdec": lm.enc = nn.ModuleList(list(lm.enc)[:k])
        else:
            lm.blocks = nn.ModuleList(list(lm.blocks)[:k])
            if bb.arch == "efficient": lm.windows = lm.windows[:k]
    set_backbone_dropout(bb, cfg["backbone_dropout"])
    apply_method_p5(bb, cfg)
    return bb, pcfg

## 4. Document representations
Every row ends with one separator token (for causal models the state there has read the whole row).

| rep | rows per document | tokens used |
|---|---|---|
| `head` | 1 | first L−1 tokens (the facts as they begin) |
| `tail` | 1 | last L−1 tokens |
| `chunks` | ≤ `n_chunks` | the document cut into consecutive chunks of `chunk_len`−1 tokens; a longer document keeps `n_chunks` of them, **spread evenly** from start to end (or the first / last ones) |

P5_01: median document 645 tokens — one 512-token row covers 43% of documents fully, 4 chunks ≈ 75%, 16 ≈ 95%.

In [7]:
def chunk_rows(toks, C, K, select="spread"):
    sep = np.array([SEP_ID]); n = max(1, math.ceil(len(toks) / C))
    idx = np.arange(n)
    if n > K:
        idx = {"first": idx[:K], "last": idx[-K:]}.get(select, np.unique(np.round(np.linspace(0, n - 1, K)).astype(int)))
    return [np.concatenate([toks[j * C:(j + 1) * C], sep]) for j in idx]

def doc_rows(toks, cfg, rep=None, n_chunks=None):
    rep = rep or cfg["rep"]; L = cfg["seq_len"]; sep = np.array([SEP_ID])
    if rep == "head": return [np.concatenate([toks[:L - 1], sep])]
    if rep == "tail": return [np.concatenate([toks[max(0, len(toks) - (L - 1)):], sep])]
    if rep == "chunks": return chunk_rows(toks, cfg["chunk_len"] - 1, n_chunks or cfg["n_chunks"], cfg["chunk_select"])
    raise ValueError(rep)

def collate(split, idxs, cfg, device, rep=None, n_chunks=None):
    rows, owner = [], []
    for b, i in enumerate(idxs):
        for r in doc_rows(split.tokens(i), cfg, rep, n_chunks): rows.append(r); owner.append(b)
    T = max(len(r) for r in rows)
    idx = np.full((len(rows), T), PAD_ID, np.int64); valid = np.zeros((len(rows), T), bool); last = np.zeros(len(rows), np.int64)
    for k, r in enumerate(rows): idx[k, :len(r)] = r; valid[k, :len(r)] = True; last[k] = len(r) - 1
    K = max(Counter(owner).values()); cidx = np.zeros((len(idxs), K), np.int64); cmask = np.zeros((len(idxs), K), bool)
    fill = Counter()
    for k, b in enumerate(owner): cidx[b, fill[b]] = k; cmask[b, fill[b]] = True; fill[b] += 1
    t = lambda a: torch.from_numpy(a).to(device, non_blocking=True)
    out = {"idx": t(idx), "valid": t(valid), "last": t(last), "cidx": t(cidx), "cmask": t(cmask)}
    if split.Y is not None: out["y"] = t(split.Y[np.asarray(idxs)].astype(np.float32))
    return out

## 5. Document model and heads
```
rows ─► backbone ─► token vectors ─┬─► [linear head]  token pooling ─► chunk pooling (attention / mean) ─► linear(100)
                                   └─► [lwan head]    label-wise attention over ALL tokens of the document's rows
```
**Label-wise attention** (LAAT, Vu et al. 2020; CAML, Mullenbach et al. 2018): `Z = tanh(W·H)`, for statute l the
attention `α_l = softmax(u_l·Z)` over the document's tokens, `v_l = Σ α_l,i h_i`, `logit_l = w_l·v_l + b_l`.
Each statute builds its own document vector from the passages relevant to it — useful when different statutes are
supported by different parts of the facts, and for rare statutes whose evidence would be drowned in one shared vector.

In [8]:
class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__(); self.w = nn.Linear(d, 1)
    def forward(self, h, mask):
        s = self.w(h).squeeze(-1).masked_fill(~mask, -1e4)
        a = torch.softmax(s, -1); return (a[..., None] * h).sum(1), a

class LabelAttention(nn.Module):
    def __init__(self, d, n_labels, da=256):
        super().__init__()
        self.proj = nn.Linear(d, da); self.U = nn.Linear(da, n_labels, bias=False)
        self.W = nn.Parameter(torch.empty(n_labels, d)); nn.init.xavier_uniform_(self.W); self.b = nn.Parameter(torch.zeros(n_labels))
    def forward(self, H, mask):                                            # H (B, N, d), mask (B, N) -> logits (B, L)
        S = self.U(torch.tanh(self.proj(H))).masked_fill(~mask[..., None], -1e4)    # (B, N, L)
        A = torch.softmax(S, 1)
        V = torch.einsum("bnl,bnd->bld", A, H)
        return (V * self.W).sum(-1) + self.b

class DocModel(nn.Module):
    def __init__(self, backbone, cfg):
        super().__init__()
        d = backbone.d; self.backbone, self.cfg = backbone, cfg
        self.prompt = None
        if cfg["method"] == "prompt":
            emb = backbone.lm.wte.weight.detach(); lo = min(1000, emb.size(0) // 2)
            self.prompt = nn.Parameter(emb[torch.randint(lo, emb.size(0), (cfg["prompt_len"],))].clone())
        self.tok_attn = AttnPool(d) if cfg["tok_pool"] == "attn" else None
        self.drop = nn.Dropout(cfg["head_dropout"])
        if cfg["head"] == "lwan": self.lwan = LabelAttention(d, N_LABELS, cfg["lwan_dim"])
        else:
            self.chunk_attn = AttnPool(d) if cfg["chunk_pool"] == "attn" else None
            self.cls = nn.Linear(d, N_LABELS)
    def token_states(self, batch):
        return self.backbone(batch["idx"], batch["valid"], self.prompt)
    def pool_rows(self, h, batch):
        m = batch["valid"]
        with torch.autocast(device_type=h.device.type, enabled=False):
            h = h.float()
            if self.cfg["tok_pool"] == "last": return h[torch.arange(h.size(0), device=h.device), batch["last"]]
            if self.tok_attn is not None: return self.tok_attn(h, m)[0]
            return (h * m[..., None]).sum(1) / m.sum(1, keepdim=True).clamp(min=1)
    def row_vectors(self, batch): return self.pool_rows(self.token_states(batch), batch)
    def forward(self, batch):
        h = self.token_states(batch)
        with torch.autocast(device_type=h.device.type, enabled=False):
            if self.cfg["head"] == "lwan":
                B, K = batch["cidx"].shape; T, d = h.size(1), h.size(2)
                Hd = h.float()[batch["cidx"]].reshape(B, K * T, d)
                mask = (batch["valid"][batch["cidx"]] & batch["cmask"][..., None]).reshape(B, K * T)
                return self.lwan(self.drop(Hd), mask)
            rv = self.pool_rows(h, batch)
            c = rv[batch["cidx"]] * batch["cmask"][..., None]; m = batch["cmask"]
            if c.size(1) == 1: v = c[:, 0]
            elif self.chunk_attn is not None: v = self.chunk_attn(c, m)[0]
            else: v = c.sum(1) / m.sum(1, keepdim=True).clamp(min=1)
            return self.cls(self.drop(v))

def build_model(cfg, device):
    bb, pcfg = build_backbone(cfg)
    return DocModel(bb, cfg).to(device), pcfg

def count_parameters(model):
    seen, total, train = set(), 0, 0
    for p in model.parameters():
        if id(p) in seen: continue
        seen.add(id(p)); total += p.numel(); train += p.numel() if p.requires_grad else 0
    return total, train

## 6. Multi-label losses (summed over the 100 statutes, averaged over documents)
Only 3.7% of (document, statute) pairs are positive, so plain BCE is dominated by easy negatives.

| loss | idea |
|---|---|
| `bce` | binary cross-entropy per statute (binary relevance) |
| `bce_pos` | positives of statute l weighted by (negatives/positives)^0.5 — rare statutes get larger weights (capped at 30) |
| `focal` (Lin et al. 2017) | `(1−p_t)^γ · BCE`: down-weights examples the model already gets right |
| `asl` (Ridnik et al. 2021) | asymmetric focusing: negatives `p_m^γ−·log(1−p_m)` with γ− = 4 and probability shift p_m = max(p − 0.05, 0) (very easy negatives are ignored, mislabelled negatives hurt less); positives plain log p |

In [9]:
def multilabel_loss(logits, y, cfg, pos_weight=None):
    kind = cfg["loss"]
    if kind in ("bce", "bce_pos"):
        return F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight if kind == "bce_pos" else None, reduction="none").sum(1).mean()
    if kind == "focal":
        ce = F.binary_cross_entropy_with_logits(logits, y, reduction="none"); p = torch.sigmoid(logits)
        pt = p * y + (1 - p) * (1 - y)
        return ((1 - pt) ** cfg["focal_gamma"] * ce).sum(1).mean()
    if kind == "asl":
        eps = 1e-8; xp = torch.sigmoid(logits); xn = 1 - xp
        if cfg["asl_clip"] > 0: xn = (xn + cfg["asl_clip"]).clamp(max=1)
        lp = y * torch.log(xp.clamp(min=eps)); ln = (1 - y) * torch.log(xn.clamp(min=eps))
        pt = xp * y + xn * (1 - y); g = cfg["asl_gamma_pos"] * y + cfg["asl_gamma_neg"] * (1 - y)
        return -((lp + ln) * (1 - pt) ** g).sum(1).mean()
    raise ValueError(kind)

def kd_loss(logits, p_teacher, T):
    """binary distillation per statute: BCE between the temperature-softened teacher and student probabilities, × T²"""
    pt = torch.sigmoid(torch.logit(p_teacher.clamp(1e-4, 1 - 1e-4)) / T)
    return F.binary_cross_entropy_with_logits(logits / T, pt, reduction="none").sum(1).mean() * T * T

## 7. Decision rules, co-occurrence stacking, metrics, error analysis
Scores → statute sets:

| rule | |
|---|---|
| `thr0.5` | p ≥ 0.5 |
| `global` | one threshold for every statute, chosen on dev |
| `per_label` | one threshold per statute chosen on dev; statutes with < 10 dev positives keep the global one |
| `…+1` | plus the most probable statute when nothing passes (every document has ≥ 1 statute) |
| `top4` | the 4 most probable statutes |

Per-statute thresholds tuned and scored on the same dev set are optimistic, so every rule is also scored
**2-fold cross-fitted** (tune on one half of dev, score the other half) and the rule used for `submission.zip` is the
one with the best cross-fitted macro-F1. **Co-occurrence stacking**: a second stage, one logistic regression per
statute on all 100 first-stage logits, learns label relationships (e.g. 12 ⇒ 13); it is trained on dev
(cross-fitted for the estimate) and applied to test — a learned post-processing, compared with independent prediction.

In [10]:
GRID = np.round(np.arange(0.02, 0.981, 0.02), 2)
def f1_grid(P, Y):
    """F1 of every statute at every grid threshold -> (G, L), zero_division = 0 (as sklearn)"""
    Yb = Y.astype(bool); out = np.zeros((len(GRID), Y.shape[1]))
    for g, t in enumerate(GRID):
        Yp = P >= t; tp = (Yp & Yb).sum(0); fp = (Yp & ~Yb).sum(0); fn = (~Yp & Yb).sum(0)
        den = 2 * tp + fp + fn; out[g] = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
    return out

def fit_rule(P, Y, rule, min_pos=10):
    if rule == "thr0.5": return {"thr": np.full(Y.shape[1], 0.5), "plus1": False}
    if rule == "top4": return {"topk": 4}
    G = f1_grid(P, Y); tg = float(GRID[int(np.argmax(G.mean(1)))]); thr = np.full(Y.shape[1], tg)
    if rule.startswith("per_label"):
        ok = Y.sum(0) >= min_pos; thr[ok] = GRID[np.argmax(G[:, ok], 0)]
    return {"thr": thr, "plus1": rule.endswith("+1")}

def apply_rule(P, r):
    if "topk" in r:
        Yp = np.zeros(P.shape, np.uint8); Yp[np.arange(len(P))[:, None], np.argsort(-P, 1)[:, :r["topk"]]] = 1; return Yp
    Yp = (P >= r["thr"][None]).astype(np.uint8)
    if r["plus1"]:
        none = Yp.sum(1) == 0; Yp[none, P[none].argmax(1)] = 1
    return Yp

RULES = ("thr0.5", "global", "global+1", "per_label", "per_label+1", "top4")
def macro(Y, Yp): return float(f1_score(Y, Yp, average="macro", zero_division=0))
def scores(Y, Yp):
    per = f1_score(Y, Yp, average=None, zero_division=0)
    out = {"macro_f1": float(per.mean()), "micro_f1": float(f1_score(Y, Yp, average="micro", zero_division=0)),
           "subset_acc": float(accuracy_score(Y, Yp)), "pred_per_doc": float(Yp.sum(1).mean())}
    for b in ("frequent", "medium", "rare"): out[f"F1_{b}"] = float(per[BUCKET == b].mean())
    return out, per

def crossfit_rules(P, Y, seed=0):
    half = np.random.default_rng(seed).permutation(len(Y)) < len(Y) // 2; res = {}
    for rule in RULES:
        Yp = np.zeros_like(Y)
        for a, b in ((half, ~half), (~half, half)): Yp[b] = apply_rule(P[b], fit_rule(P[a], Y[a], rule))
        res[rule] = macro(Y, Yp)
    return res

def stack_probs(P_dev, Y_dev, P_test, seed=0, C=1.0):
    """per-statute logistic regression on the 100 first-stage logits; returns (cross-fitted dev probs, test probs)"""
    def feats(P): return np.log(np.clip(P, 1e-5, 1 - 1e-5) / (1 - np.clip(P, 1e-5, 1 - 1e-5)))
    Xd, Xt = feats(P_dev), feats(P_test)
    def fit_predict(Xa, Ya, Xb):
        out = np.zeros((len(Xb), Ya.shape[1]), np.float32)
        for l in range(Ya.shape[1]):
            if Ya[:, l].min() == Ya[:, l].max(): out[:, l] = Ya[0, l]; continue
            out[:, l] = LogisticRegression(C=C, max_iter=300).fit(Xa, Ya[:, l]).predict_proba(Xb)[:, 1]
        return out
    half = np.random.default_rng(seed).permutation(len(Y_dev)) < len(Y_dev) // 2; cf = np.zeros(P_dev.shape, np.float32)
    for a, b in ((half, ~half), (~half, half)): cf[b] = fit_predict(Xd[a], Y_dev[a], Xd[b])
    return cf, fit_predict(Xd, Y_dev, Xt)

def error_analysis(split, Yp, per):
    Y = split.Y; fp = (Yp == 1) & (Y == 0); fn = (Yp == 0) & (Y == 1)
    conf = Counter()
    for i in np.where(fn.any(1) & fp.any(1))[0]:
        for a in np.where(fn[i])[0]:
            for b in np.where(fp[i])[0]: conf[(int(a), int(b))] += 1
    card = Y.sum(1); ex_f1 = 2 * (Yp & Y).sum(1) / np.maximum(Yp.sum(1) + card, 1)
    groups = {"1 statute": card == 1, "2-3": (card >= 2) & (card <= 3), "4-6": (card >= 4) & (card <= 6), "7+": card >= 7}
    q = np.quantile(split.lengths, [0, .25, .5, .75, 1])
    return {"most_false_positives": [(int(i), int(fp[:, i].sum())) for i in np.argsort(-fp.sum(0))[:10]],
            "most_missed": [(int(i), int(fn[:, i].sum())) for i in np.argsort(-fn.sum(0))[:10]],
            "missed_a_predicted_b": [(a, b, c) for (a, b), c in conf.most_common(10)],
            "worst_statutes": [(int(i), round(float(per[i]), 3), int(Y[:, i].sum())) for i in np.argsort(per)[:10]],
            "example_F1_by_statute_count": {k: round(float(ex_f1[m].mean()), 4) for k, m in groups.items() if m.any()},
            "micro_F1_by_length_quartile": {f"Q{j+1} ({int(q[j])}-{int(q[j+1])} tok)": round(float(f1_score(
                Y[(split.lengths >= q[j]) & (split.lengths <= q[j + 1])], Yp[(split.lengths >= q[j]) & (split.lengths <= q[j + 1])],
                average="micro", zero_division=0)), 4) for j in range(4)},
            "spearman_F1_vs_train_freq": float(pd.Series(per).corr(pd.Series(CNT_TRAIN), method="spearman"))}

def write_submission(ids, Yp, d, name):
    obj = [{"id": i, "pred": [int(x) for x in np.where(row)[0]]} for i, row in zip(ids, Yp)]
    assert len(obj) == len(SPLITS["test"]) and {o["id"] for o in obj} == set(SPLITS["test"].ids)
    assert all(0 <= x < N_LABELS for o in obj for x in o["pred"])
    with zipfile.ZipFile(d / name, "w", zipfile.ZIP_DEFLATED) as z: z.writestr("prediction.json", json.dumps(obj))

@torch.no_grad()
def predict(model, split, cfg, device, idxs=None, batch=None):
    model.eval(); idxs = np.arange(len(split)) if idxs is None else idxs; out = np.zeros((len(idxs), N_LABELS), np.float32)
    use_amp = device.type == "cuda"; bs = batch or max(4, cfg["batch"] * 2)
    for i in range(0, len(idxs), bs):
        b = collate(split, idxs[i:i + bs], cfg, device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(b)
        out[i:i + bs] = torch.sigmoid(logits.float()).cpu().numpy()
    return out

def dev_macro_global(P, Y):
    G = f1_grid(P, Y); g = int(np.argmax(G.mean(1))); return float(G[g].mean()), float(GRID[g])

## 8. Training (+ knowledge distillation), experiment runner
AdamW, warmup + linear decay, fp16 autocast, gradient accumulation. After every epoch the dev set is scored (macro-F1
with the best global threshold) and the best epoch's trainable weights are kept.
**KD**: the teacher's 100 probabilities for every training document (computed once, with the teacher's own
representation) are soft targets: `α·task loss + (1−α)·T²·BCE(σ(z_s/T), σ(z_t/T))`, α = 0.5, T = 2.

In [11]:
def subset_indices(split, n, seed):
    if not n or n >= len(split): return np.arange(len(split))
    return np.sort(np.random.default_rng(seed).choice(len(split), n, replace=False))

def load_trained(name, device):
    cp, w = find_exp_file(name, "config.json"), find_exp_file(name, "trainable.pt")
    assert cp is not None and w is not None, f"p5/exps/{name}/(config.json, trainable.pt) not found — attach the output of the run that trained {name}"
    tcfg = dict(DEFAULTS); tcfg.update(load_json(cp))
    m, _ = build_model(tcfg, device)
    sd = torch.load(w, map_location="cpu", weights_only=False)
    res = m.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in sd.items()}, strict=False)
    assert not res.unexpected_keys, res.unexpected_keys
    return m, tcfg

def pos_weight_tensor(Y, power, device):
    pos = Y.sum(0).astype(np.float64); neg = len(Y) - pos
    return torch.tensor(np.clip((neg / np.maximum(pos, 1)) ** power, 1, 30), dtype=torch.float32, device=device)

def train_model(cfg, device, log):
    split = SPLITS["train"]; tr_idx = subset_indices(split, cfg["max_train_docs"], cfg["seed"])
    soft = None; teacher_params = None
    if cfg["teacher"]:
        teacher, tcfg = load_trained(cfg["teacher"], device); teacher_params = count_parameters(teacher)[0]
        t0 = time.time(); p = predict(teacher, split, tcfg, device, tr_idx)
        soft = np.zeros((len(split), N_LABELS), np.float16); soft[tr_idx] = p
        log(f"teacher {cfg['teacher']} ({teacher_params/1e6:.1f}M): soft targets for {len(tr_idx)} docs in {time.time()-t0:.0f}s")
        del teacher
        if device.type == "cuda": torch.cuda.empty_cache()
    model, pcfg = build_model(cfg, device); total, trainable = count_parameters(model)
    if cfg["method"] == "gradual":
        n_blocks = len(list(model.backbone.blocks())); gradual_unfreeze(model.backbone, 1); trainable = count_parameters(model)[1]
    hp = [p for n, p in model.named_parameters() if not n.startswith("backbone.") and n != "prompt"]
    bp = [p for n, p in model.named_parameters() if n.startswith("backbone.") and (p.requires_grad or cfg["method"] == "gradual")]
    groups = [{"params": [p for p in hp if p.ndim >= 2], "lr": cfg["lr_head"], "weight_decay": cfg["weight_decay"]},
              {"params": [p for p in hp if p.ndim < 2], "lr": cfg["lr_head"], "weight_decay": 0.0}]
    if model.prompt is not None: groups.append({"params": [model.prompt], "lr": cfg["lr"], "weight_decay": 0.0})
    elif bp and cfg["lr"] > 0:
        groups += [{"params": [p for p in bp if p.ndim >= 2], "lr": cfg["lr"], "weight_decay": cfg["weight_decay"]},
                   {"params": [p for p in bp if p.ndim < 2], "lr": cfg["lr"], "weight_decay": 0.0}]
    opt = torch.optim.AdamW([g for g in groups if g["params"]], betas=(0.9, 0.98), eps=1e-8)
    for g in opt.param_groups: g["base_lr"] = g["lr"]
    upd_per_epoch = math.ceil(len(tr_idx) / (cfg["batch"] * cfg["accum"])); total_upd = max(1, upd_per_epoch * cfg["epochs"])
    warm = max(1, int(cfg["warmup"] * total_upd))
    use_amp = device.type == "cuda"; scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    pw = pos_weight_tensor(split.Y[tr_idx], cfg["pos_weight_power"], device)
    log(f"model {cfg['model']} ({pcfg['arch']}{', ' + str(cfg['keep_layers']) + ' layers' if cfg['keep_layers'] else ''}) attn={cfg['attn']} "
        f"method={cfg['method']} rep={cfg['rep']}({cfg['seq_len'] if cfg['rep'] != 'chunks' else str(cfg['n_chunks']) + 'x' + str(cfg['chunk_len'])}) "
        f"head={cfg['head']} loss={cfg['loss']} | params {total/1e6:.1f}M, trainable {trainable/1e6:.3f}M ({100*trainable/total:.2f}%) | "
        f"{len(tr_idx)} train docs, {total_upd} updates")
    if use_amp: torch.cuda.reset_peak_memory_stats(device)
    dev = SPLITS["dev"]; rng = np.random.default_rng(cfg["seed"]); hist, best = [], (-1.0, None, 0)
    upd, t0 = 0, time.time()
    for ep in range(1, cfg["epochs"] + 1):
        model.train(); order = rng.permutation(tr_idx); run, nb = 0.0, 0
        opt.zero_grad(set_to_none=True)
        for i in range(0, len(order), cfg["batch"]):
            if cfg["method"] == "gradual":
                gradual_unfreeze(model.backbone, min(n_blocks, 1 + (upd * n_blocks) // max(1, int(0.6 * total_upd))))
            part = order[i:i + cfg["batch"]]; b = collate(split, part, cfg, device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(b)
            logits = logits.float(); loss = multilabel_loss(logits, b["y"], cfg, pw)
            if soft is not None:
                a = cfg["kd_alpha"]; loss = a * loss + (1 - a) * kd_loss(logits, torch.from_numpy(soft[part].astype(np.float32)).to(device), cfg["kd_T"])
            scaler.scale(loss / cfg["accum"]).backward(); run += loss.item(); nb += 1
            if nb % cfg["accum"] == 0 or i + cfg["batch"] >= len(order):
                lr_mult = (upd + 1) / warm if upd < warm else max(0.0, (total_upd - upd) / max(1, total_upd - warm))
                for g in opt.param_groups: g["lr"] = g["base_lr"] * lr_mult
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"] if p.grad is not None], cfg["grad_clip"])
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); upd += 1
        P = predict(model, dev, cfg, device); f1g, tg = dev_macro_global(P, dev.Y)
        msg = {"epoch": ep, "loss": run / max(nb, 1), "min": (time.time() - t0) / 60, "dev_macro@0.5": macro(dev.Y, (P >= 0.5).astype(np.uint8)),
               "dev_macro_global": f1g, "thr": tg}
        if f1g > best[0]:
            best = (f1g, {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters()
                          if p.requires_grad or not n.startswith("backbone.") or cfg["method"] in ("gradual", "full")}, ep)
        hist.append(msg); log(json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in msg.items()}))
    model.load_state_dict(best[1], strict=False)
    info = {"total_params": total, "trainable_params": trainable, "train_min": (time.time() - t0) / 60, "history": hist,
            "peak_mem_gb": torch.cuda.max_memory_allocated(device) / 1e9 if use_amp else None, "best_epoch": best[2],
            "backbone_arch": pcfg["arch"], "teacher_params": teacher_params, "train_docs": int(len(tr_idx))}
    return model, info, set(best[1])

def save_trainable(model, names, path):
    sd = {n: p.detach().half().cpu() if p.is_floating_point() else p.detach().cpu() for n, p in model.named_parameters() if n in names}
    torch.save(sd, str(path) + ".tmp"); os.replace(str(path) + ".tmp", path)

def finish(name, cfg, d, P_dev, P_test, info, log):
    dev = SPLITS["dev"]; Y = dev.Y
    cf = crossfit_rules(P_dev, Y)
    on_dev = {r: macro(Y, apply_rule(P_dev, fit_rule(P_dev, Y, r))) for r in RULES}
    t0 = time.time(); S_dev_cf, S_test = stack_probs(P_dev, Y, P_test)
    cf_stack = crossfit_rules(S_dev_cf, Y)["global+1"]; stack_min = (time.time() - t0) / 60
    cands = {r: cf[r] for r in ("global", "global+1", "per_label", "per_label+1")}; cands["stacked+global+1"] = cf_stack
    rule = max(cands, key=cands.get)
    if rule == "stacked+global+1":
        r_fit = fit_rule(S_dev_cf, Y, "global+1"); Yd, Yt = apply_rule(S_dev_cf, r_fit), apply_rule(S_test, r_fit)
    else:
        r_fit = fit_rule(P_dev, Y, rule); Yd, Yt = apply_rule(P_dev, r_fit), apply_rule(P_test, r_fit)
    main, per = scores(Y, Yd)
    rg = fit_rule(P_dev, Y, "global+1"); g_scores, _ = scores(Y, apply_rule(P_dev, rg))
    write_submission(SPLITS["test"].ids, Yt, d, "submission.zip")
    write_submission(SPLITS["test"].ids, apply_rule(P_test, rg), d, "submission_global.zip")
    if rule != "stacked+global+1":
        rs = fit_rule(S_dev_cf, Y, "global+1"); write_submission(SPLITS["test"].ids, apply_rule(S_test, rs), d, "submission_stacked.zip")
    ea = error_analysis(dev, Yd, per)
    save_json({"chosen_rule": rule, "main_on_dev": main, "global+1_on_dev": g_scores, "crossfit_macro": cf, "crossfit_stacked_global+1": cf_stack,
               "fit_on_dev_macro (optimistic)": on_dev, "per_statute_F1": [round(float(x), 4) for x in per], "thresholds": np.round(r_fit.get("thr", np.zeros(1)), 2).tolist(),
               "errors": ea}, d / "dev_metrics.json")
    np.save(d / "prob_dev.npy", P_dev.astype(np.float16)); np.save(d / "prob_test.npy", P_test.astype(np.float16))
    row = {"name": name, "model": cfg["model"] + (f"[{cfg['keep_layers']}L]" if cfg.get("keep_layers") else ""), "attn": cfg["attn"], "init": cfg["init"],
           "method": cfg["method"] + (f"-{cfg['lorma_variant']}" if cfg["method"] == "lorma" else "") + (f"+KD({cfg['teacher']})" if cfg["teacher"] else ""),
           "rep": cfg["rep"] + (f"{cfg['seq_len']}" if cfg["rep"] != "chunks" else f"{cfg['n_chunks']}x{cfg['chunk_len']}"),
           "head": cfg["head"], "loss": cfg["loss"], "rule": rule, "dev_macro": main["macro_f1"], "dev_macro_cf": cands[rule],
           "cf_global+1": cf["global+1"], "cf_per_label+1": cf["per_label+1"], "cf_stacked": cf_stack, "cf_thr0.5": cf["thr0.5"], "cf_top4": cf["top4"],
           "dev_micro": main["micro_f1"], "dev_subset": main["subset_acc"], "F1_frequent": main["F1_frequent"], "F1_medium": main["F1_medium"],
           "F1_rare": main["F1_rare"], "pred_per_doc": main["pred_per_doc"], "train_docs": info.get("train_docs"),
           "total_params_M": info["total_params"] / 1e6, "trainable_params_M": info["trainable_params"] / 1e6,
           "trainable_%": 100 * info["trainable_params"] / info["total_params"], "teacher_params_M": (info.get("teacher_params") or 0) / 1e6,
           "peak_mem_GB": info.get("peak_mem_gb"), "train_min": info["train_min"], "infer_ms_per_doc": info.get("infer_ms_per_doc"),
           "best_epoch": info.get("best_epoch")}
    save_json({"row": row, "info": info, "cfg": cfg}, d / "result.json")
    log(f"DONE dev macro-F1 {main['macro_f1']:.4f} (rule {rule}; cross-fitted {cands[rule]:.4f}) | micro {main['micro_f1']:.4f} | "
        f"frequent/medium/rare {main['F1_frequent']:.3f}/{main['F1_medium']:.3f}/{main['F1_rare']:.3f} | cross-fitted: "
        f"thr0.5 {cf['thr0.5']:.4f}, global+1 {cf['global+1']:.4f}, per_label+1 {cf['per_label+1']:.4f}, stacked {cf_stack:.4f} (stacking {stack_min:.1f} min)")
    return row

def run_experiment(name, device):
    cfg = exp_cfg(name); d = EXPS_DIR / name; d.mkdir(parents=True, exist_ok=True)
    if (d / "result.json").exists(): print(f"[{name}] already done — skipped"); return
    logf = open(d / "log.txt", "a")
    def log(msg):
        line = f"[{name} | {device}] {msg}"; print(line, flush=True); logf.write(line + "\n"); logf.flush()
    save_json(cfg, d / "config.json")
    if cfg["hier_from"]:
        run_hierarchical(name, cfg, d, device, log); logf.close(); return
    model, info, names = train_model(cfg, device, log)
    save_trainable(model, names, d / "trainable.pt")
    t0 = time.time(); P_dev = predict(model, SPLITS["dev"], cfg, device)
    info["infer_ms_per_doc"] = 1000 * (time.time() - t0) / len(SPLITS["dev"])
    P_test = predict(model, SPLITS["test"], cfg, device)
    del model
    if device.type == "cuda": torch.cuda.empty_cache()
    finish(name, cfg, d, P_dev, P_test, info, log); logf.close()

## 9. Hierarchical two-stage model (`hier_from=<finished experiment>`)
1. The finished experiment's fine-tuned encoder embeds up to `hier_chunks` (16) chunks of 512 tokens of every
   document (spread over the document when it is longer) with its own token pooling → (docs, 16, d). 16 × 511 tokens
   cover the whole document for ≈ 95% of cases. Inference only — far cheaper than training through all chunks.
2. A small model: LayerNorm → + chunk-position embedding → BiGRU → **label-wise attention over the chunks** (each
   statute picks the chunks that support it) → 100 logits; same loss as the encoder, best epoch on dev.

In [12]:
@torch.no_grad()
def embed_chunks(enc, split, cfg, K, device, idxs=None, rows_per_batch=32):
    enc.eval(); idxs = np.arange(len(split)) if idxs is None else idxs; d = enc.backbone.d
    E = np.zeros((len(idxs), K, d), np.float16); M = np.zeros((len(idxs), K), bool); use_amp = device.type == "cuda"
    docs_per = max(1, rows_per_batch // K)
    for i in range(0, len(idxs), docs_per):
        part = idxs[i:i + docs_per]; b = collate(split, part, dict(cfg, chunk_select="spread"), device, rep="chunks", n_chunks=K)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            rv = enc.row_vectors(b)
        c = (rv[b["cidx"]] * b["cmask"][..., None]).float().cpu().numpy(); m = b["cmask"].cpu().numpy()
        k = c.shape[1]; E[i:i + len(part), :k] = c; M[i:i + len(part), :k] = m
    return E, M

class HierHead(nn.Module):
    def __init__(self, d, head="lwan", h=256):
        super().__init__()
        self.inp = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.1)); self.pos = nn.Parameter(torch.zeros(1, 64, d))
        self.gru = nn.GRU(d, h, batch_first=True, bidirectional=True); self.drop = nn.Dropout(0.2)
        if head == "lwan": self.lwan = LabelAttention(2 * h, N_LABELS, 128); self.att = None
        else: self.att = AttnPool(2 * h); self.cls = nn.Linear(2 * h, N_LABELS)
    def forward(self, x, m):
        x = self.inp(x + self.pos[:, :x.size(1)]); x, _ = self.gru(x); x = self.drop(x)
        if self.att is None: return self.lwan(x, m)
        return self.cls(self.att(x, m)[0])

def run_hierarchical(name, cfg, d, device, log):
    enc, ecfg = load_trained(cfg["hier_from"], device)
    ecfg = dict(ecfg, chunk_len=ecfg["seq_len"] if ecfg["rep"] != "chunks" else ecfg["chunk_len"])
    K = cfg["hier_chunks"]; tr_idx = subset_indices(SPLITS["train"], cfg["max_train_docs"], cfg["seed"])
    t0 = time.time()
    Etr, Mtr = embed_chunks(enc, SPLITS["train"], ecfg, K, device, tr_idx)
    Edv, Mdv = embed_chunks(enc, SPLITS["dev"], ecfg, K, device); Ete, Mte = embed_chunks(enc, SPLITS["test"], ecfg, K, device)
    emb_min = (time.time() - t0) / 60; log(f"embedded ≤ {K} chunks x {ecfg['chunk_len']} tokens for {len(tr_idx)}+{len(Edv)}+{len(Ete)} docs in {emb_min:.1f} min")
    enc_params = count_parameters(enc)[0]; del enc
    if device.type == "cuda": torch.cuda.empty_cache()
    torch.manual_seed(cfg["seed"]); head = HierHead(Etr.shape[-1], cfg["hier_head"]).to(device)
    opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=0.01)
    Ytr = SPLITS["train"].Y[tr_idx]; pw = pos_weight_tensor(Ytr, ecfg["pos_weight_power"], device); lcfg = dict(ecfg)
    T = lambda a: torch.from_numpy(np.ascontiguousarray(a)).to(device)
    def probs(E, M):
        head.eval(); out = []
        with torch.no_grad():
            for i in range(0, len(E), 256): out.append(torch.sigmoid(head(T(E[i:i + 256]).float(), T(M[i:i + 256]))).cpu().numpy())
        return np.concatenate(out)
    steps = cfg["hier_epochs"] * math.ceil(len(tr_idx) / 64); sched = torch.optim.lr_scheduler.OneCycleLR(opt, 1e-3, total_steps=steps, pct_start=0.1)
    rng = np.random.default_rng(cfg["seed"]); best = (-1, None, 0); t1 = time.time()
    for ep in range(1, cfg["hier_epochs"] + 1):
        head.train(); order = rng.permutation(len(tr_idx))
        for i in range(0, len(order), 64):
            j = order[i:i + 64]; logits = head(T(Etr[j]).float(), T(Mtr[j]))
            loss = multilabel_loss(logits, T(Ytr[j].astype(np.float32)), lcfg, pw); opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0); opt.step(); sched.step()
        f1g, tg = dev_macro_global(probs(Edv, Mdv), SPLITS["dev"].Y)
        log(json.dumps({"epoch": ep, "dev_macro_global": round(f1g, 4), "thr": tg}))
        if f1g > best[0]: best = (f1g, {k: v.detach().clone() for k, v in head.state_dict().items()}, ep)
    head.load_state_dict(best[1])
    P_dev, P_test = probs(Edv, Mdv), probs(Ete, Mte)
    info = {"total_params": enc_params + count_parameters(head)[0], "trainable_params": count_parameters(head)[0],
            "train_min": emb_min + (time.time() - t1) / 60, "embed_min": emb_min, "best_epoch": best[2], "train_docs": int(len(tr_idx)),
            "backbone_arch": ecfg["model"], "chunks_used_mean_dev": float(Mdv.sum(1).mean())}
    cfg_row = dict(cfg, model=ecfg["model"], attn=ecfg["attn"], method=f"hier({cfg['hier_from']})", rep="chunks", n_chunks=K,
                   chunk_len=ecfg["chunk_len"], head=f"hier-{cfg['hier_head']}", loss=ecfg["loss"], keep_layers=ecfg.get("keep_layers"))
    finish(name, cfg_row, d, P_dev, P_test, info, log)

## 10. Unit tests (CPU, seconds)

In [13]:
def _tests():
    cfg = dict(DEFAULTS, seq_len=16, chunk_len=8, n_chunks=3)
    toks = np.arange(100, 150)                                             # 50 tokens -> 8 chunks of 7
    r = doc_rows(toks, cfg, "head")[0]; assert len(r) == 16 and r[-1] == SEP_ID and (r[:-1] == toks[:15]).all()
    r = doc_rows(toks, cfg, "tail")[0]; assert (r[:-1] == toks[-15:]).all()
    rows = doc_rows(toks, cfg, "chunks"); assert len(rows) == 3 and (rows[0][:-1] == toks[:7]).all() and (rows[-1][:-1] == toks[49:]).all(), "spread: first and last chunk"
    assert len(chunk_rows(toks[:10], 7, 3)) == 2 and len(chunk_rows(np.zeros(0, np.int64), 7, 3)) == 1
    assert (np.concatenate([x[:-1] for x in chunk_rows(toks[:20], 7, 4)]) == toks[:20]).all()
    # label attention ignores masked tokens
    la = LabelAttention(6, 5, 4); H = torch.randn(2, 7, 6); m = torch.ones(2, 7, dtype=torch.bool); m[:, 5:] = False
    H2 = H.clone(); H2[:, 5:] = 100.0; assert torch.allclose(la(H, m), la(H2, m), atol=1e-5)
    # vectorised F1 = sklearn macro-F1 (zero_division=0)
    rng = np.random.default_rng(1); Y = (rng.random((60, 7)) < 0.2).astype(np.uint8); Y[:, 6] = 0; P = rng.random((60, 7))
    G = f1_grid(P, Y); g = 10; assert abs(G[g].mean() - macro(Y, (P >= GRID[g]).astype(np.uint8))) < 1e-9
    rp = fit_rule(P, Y, "per_label+1", min_pos=1); Yp = apply_rule(P, rp); assert (Yp.sum(1) >= 1).all()
    # ASL with no focusing / no clip = BCE; focal with gamma 0 = BCE
    z = torch.randn(4, 7); yt = torch.from_numpy(Y[:4].astype(np.float32))
    c0 = dict(DEFAULTS, loss="asl", asl_gamma_neg=0.0, asl_gamma_pos=0.0, asl_clip=0.0)
    assert abs(multilabel_loss(z, yt, c0).item() - multilabel_loss(z, yt, dict(DEFAULTS, loss="bce")).item()) < 1e-4
    assert abs(multilabel_loss(z, yt, dict(DEFAULTS, loss="focal", focal_gamma=0.0)).item() - multilabel_loss(z, yt, dict(DEFAULTS, loss="bce")).item()) < 1e-5
    assert kd_loss(z, torch.sigmoid(z), 2.0).item() <= kd_loss(z, torch.sigmoid(-z), 2.0).item()
    class _Stack(nn.Module):
        def __init__(self): super().__init__(); self.bl = nn.ModuleList([nn.Linear(2, 2) for _ in range(4)])
        def blocks(self): return self.bl
    st = _Stack(); gradual_unfreeze(st, 1); assert [b.weight.requires_grad for b in st.bl] == [False, False, False, True]
    base = nn.Linear(16, 24, bias=False); x = torch.randn(5, 16)
    for mod in (LoRMALinear(base, 4, 8), AdapterLinear(base, 6), BiasLinear(base)): assert torch.allclose(mod(x), base(x), atol=1e-5)
    print("unit tests passed")
_tests()

unit tests passed


## 11. Run the queue (one worker thread per GPU; experiments wait for an encoder / teacher trained in the same run)

In [14]:
def run_queue(names):
    missing = []
    for n in names:
        c = exp_cfg(n); dep = c["hier_from"] or c["teacher"]
        if dep and dep not in names and not (EXPS_DIR / n / "result.json").exists() and find_exp_file(dep, "trainable.pt") is None:
            missing.append(f"{n} needs {dep}")
    if missing: print("WARNING — these will fail, attach the runs that trained the listed experiments:\n  " + "\n  ".join(missing))
    pending, running, failed, lock = list(names), set(), set(), threading.Lock()
    def dep_of(n): c = exp_cfg(n); return c["hier_from"] or c["teacher"]
    def take():
        with lock:
            for n in pending:
                dep = dep_of(n)
                if dep in failed:
                    pending.remove(n); failed.add(n); print(f"[{n}] not run: its dependency {dep} failed"); return "retry"
                if dep and (dep in pending or dep in running): continue
                pending.remove(n); running.add(n); return n
            return "wait" if pending else None
    devices = [torch.device(f"cuda:{i}") for i in range(NUM_GPUS)] or [torch.device("cpu")]
    def worker(dev):
        while True:
            name = take()
            if name is None: return
            if name in ("wait", "retry"):
                if name == "wait": time.sleep(20)
                continue
            ok, need = False, exp_cfg(name)["est_h"]
            if not (EXPS_DIR / name / "result.json").exists() and time_left_h() < need + 0.25:
                print(f"[{name}] skipped: needs ≈ {need:.1f} h, {time_left_h():.2f} h left — run it in a new session (finished ones are kept)")
            else:
                try:
                    if dev.type == "cuda":
                        with torch.cuda.device(dev): run_experiment(name, dev)
                    else: run_experiment(name, dev)
                    ok = (EXPS_DIR / name / "result.json").exists()
                except Exception:
                    print(f"[{name}] FAILED:\n{traceback.format_exc()}", flush=True)
                    (EXPS_DIR / name).mkdir(parents=True, exist_ok=True); (EXPS_DIR / name / "error.txt").write_text(traceback.format_exc())
            with lock:
                running.discard(name)
                if not ok: failed.add(name)
    threads = [threading.Thread(target=worker, args=(dv,)) for dv in devices]
    for t in threads: t.start()
    for t in threads: t.join()

print("estimated hours:", {n: round(exp_cfg(n)["est_h"], 1) for n in RUN_QUEUE})
t_queue = time.time(); run_queue(RUN_QUEUE); print(f"queue finished in {(time.time() - t_queue)/60:.1f} min")

estimated hours: {'s2_A0': 1.4, 's2_A1': 1.4, 's2_F': 1.4, 's2_B': 1.4}
[s2_A0 | cuda:0] model A0 (gpt2) attn=causal method=full rep=head(512) head=lwan loss=asl | params 111.3M, trainable 111.308M (100.00%) | 42207 train docs, 7914 updates
[s2_A1 | cuda:1] model A1 (modern) attn=causal method=full rep=head(512) head=lwan loss=asl | params 110.4M, trainable 110.419M (100.00%) | 42207 train docs, 7914 updates
[s2_A0 | cuda:0] {"epoch": 1, "loss": 3.4198, "min": 17.3249, "dev_macro@0.5": 0.3012, "dev_macro_global": 0.3174, "thr": 0.56}
[s2_A1 | cuda:1] {"epoch": 1, "loss": 3.2746, "min": 25.8054, "dev_macro@0.5": 0.308, "dev_macro_global": 0.3173, "thr": 0.54}
[s2_A0 | cuda:0] {"epoch": 2, "loss": 2.7292, "min": 34.7106, "dev_macro@0.5": 0.3319, "dev_macro_global": 0.3522, "thr": 0.56}
[s2_A1 | cuda:1] {"epoch": 2, "loss": 2.6759, "min": 51.6892, "dev_macro@0.5": 0.3346, "dev_macro_global": 0.3512, "thr": 0.56}
[s2_A0 | cuda:0] {"epoch": 3, "loss": 2.5057, "min": 52.1064, "dev_macro@0.5"

## 12. Results (this run + attached earlier runs)
`dev_macro` = official macro-F1 on clean dev with the chosen rule (fitted on all of dev — slightly optimistic);
`dev_macro_cf` = the same rule cross-fitted (the honest estimate used to choose it); `cf_*` = cross-fitted macro-F1
of the other rules; `F1_frequent/medium/rare` = mean F1 of the statutes in each third by training frequency.

In [15]:
rows = [load_json(f)["row"] for f in sorted(EXPS_DIR.glob("*/result.json"))]
if rows:
    df = pd.DataFrame(rows).sort_values("dev_macro_cf", ascending=False)
    df.to_csv(ROOT / "p5" / "p5_experiments.csv", index=False)
    cols = ["name", "model", "method", "rep", "head", "loss", "rule", "dev_macro_cf", "dev_macro", "cf_thr0.5", "cf_global+1", "cf_per_label+1",
            "cf_stacked", "dev_micro", "dev_subset", "F1_frequent", "F1_medium", "F1_rare", "pred_per_doc", "trainable_%", "peak_mem_GB",
            "train_min", "infer_ms_per_doc", "best_epoch"]
    display(df[[c for c in cols if c in df]].round(4).to_string(index=False))
    best = df.iloc[0]["name"]; m = load_json(EXPS_DIR / best / "dev_metrics.json")
    print(f"\nbest: {best} -> {ROOT}/p5/exps/{best}/submission.zip")
    print("error analysis of the best:", json.dumps(m["errors"], indent=1))
for f in sorted(EXPS_DIR.glob("*/error.txt")): print(f"FAILED: {f.parent.name}\n{f.read_text()[-1500:]}")

' name model method     rep head loss        rule  dev_macro_cf  dev_macro  cf_thr0.5  cf_global+1  cf_per_label+1  cf_stacked  dev_micro  dev_subset  F1_frequent  F1_medium  F1_rare  pred_per_doc  trainable_%  peak_mem_GB  train_min  infer_ms_per_doc  best_epoch\ns2_A1    A1   full head512 lwan  asl per_label+1        0.3779     0.3891     0.3497       0.3698          0.3779      0.3376     0.4844      0.0458       0.4955     0.3654   0.3031        4.8502        100.0       4.3056    77.7131            9.6940           3\ns2_A0    A0   full head512 lwan  asl   per_label        0.3744     0.3883     0.3483       0.3689          0.3743      0.3488     0.4831      0.0390       0.4962     0.3626   0.3027        4.8375        100.0       3.5189    52.1135            6.3681           3\n s2_F     F   full head512 lwan  asl per_label+1        0.3742     0.3876     0.3494       0.3704          0.3742      0.3342     0.4886      0.0461       0.4898     0.3655   0.3045        4.4942        100.


best: s2_A1 -> /kaggle/working/cs781_p1/p5/exps/s2_A1/submission.zip
error analysis of the best: {
 "most_false_positives": [
  [
   43,
   1715
  ],
  [
   5,
   1678
  ],
  [
   97,
   1672
  ],
  [
   39,
   1248
  ],
  [
   46,
   932
  ],
  [
   51,
   886
  ],
  [
   44,
   831
  ],
  [
   82,
   685
  ],
  [
   35,
   676
  ],
  [
   10,
   657
  ]
 ],
 "most_missed": [
  [
   5,
   787
  ],
  [
   43,
   676
  ],
  [
   39,
   634
  ],
  [
   97,
   634
  ],
  [
   10,
   505
  ],
  [
   12,
   477
  ],
  [
   44,
   468
  ],
  [
   34,
   467
  ],
  [
   13,
   463
  ],
  [
   14,
   396
  ]
 ],
 "missed_a_predicted_b": [
  [
   13,
   5,
   160
  ],
  [
   12,
   5,
   146
  ],
  [
   14,
   5,
   141
  ],
  [
   39,
   43,
   118
  ],
  [
   39,
   5,
   114
  ],
  [
   10,
   5,
   111
  ],
  [
   39,
   97,
   108
  ],
  [
   5,
   97,
   103
  ],
  [
   97,
   5,
   103
  ],
  [
   13,
   43,
   100
  ]
 ],
 "worst_statutes": [
  [
   0,
   0.105,
   86
  ],
  [
   8,
  